In [ ]:
import warnings
warnings.filterwarnings("ignore")

#### General Guide for ML implementation in this study

1. Load data

2. Basic EDA (info, describe)

3. Log-transform target (creep life)

4. Define X and y_log

5. Train–validation–test split

6. Fit Min–Max scaler on X_train only

7. Transform X_val, X_test using same scaler

#### Data loading 

In [ ]:
import pandas as pd 
import numpy as np 

uncreep=pd.read_excel("Combined - edited.xlsx") 

In [ ]:
uncreep.info()

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
uncreep.describe(include='all')

In [ ]:
import pandas as pd

# Display options (optional, for notebook viewing)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Generate describe table
desc = uncreep.describe(include='all')

# Transpose so features are rows and statistics are columns
desc_T = desc.T

# Optional: reset index to make a clean 'variable' column
desc_T = desc_T.reset_index().rename(columns={'index': 'variable'})

# Optional: round numeric values for cleaner Excel
desc_T = desc_T.round(4)

# Export to Excel
output_file = "Unified_dataset_statistics.xlsx"
desc_T.to_excel(output_file, index=False)

print(f"Describe table exported cleanly to: {output_file}")


#### Preprocessing

In [ ]:
uncreep["log_life"] = np.log(uncreep["Time to rupture/h"])

In [ ]:
X=uncreep.drop(columns=["log_life","Time to rupture/h"])
y=uncreep["log_life"]

#### BNN - MCD Code 

In [ ]:
# ============================================================
#  BNN WITH MONTE CARLO DROPOUT
#
#  Key methodological improvements over original:
#  1. Architecture selected by 5-fold CV on training data
#     using mean NLL as the sole criterion
#  2. Repeated evaluation over 10 random seeds
#     → all metrics reported as mean ± std
#  3. Test set is never touched until final evaluation
# ============================================================

import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats

import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ============================================================
#  SECTION 1 — CONFIGURATION
#  All experiment settings are defined here in one place.
# ============================================================

# 10 seeds for repeated evaluation 
SEEDS = [42, 7, 13, 21, 37, 55, 71, 88, 99, 123]

# Candidate architectures for selection
# Each entry: layer sizes and corresponding dropout rates
CANDIDATE_ARCHITECTURES = {
    "[32, 16, 8]":  {"layers": [32, 16, 8],  "dropout": [0.20, 0.15, 0.10]},
    "[48, 32, 16]": {"layers": [48, 32, 16], "dropout": [0.25, 0.20, 0.15]},
    "[64, 32, 16]": {"layers": [64, 32, 16], "dropout": [0.30, 0.20, 0.10]},
}

# Cross-validation settings
N_FOLDS       = 5     # number of CV folds for architecture selection
MC_SAMPLES_CV = 100   # MC samples during CV (faster); 500 for final evaluation
MC_SAMPLES    = 500   # MC samples for final test evaluation

# Training settings
EPOCHS_CV    = 500    # max epochs during CV fold training
EPOCHS_FINAL = 1000   # max epochs for final model training
BATCH_SIZE   = 16
LEARNING_RATE = 0.001
NOMINAL_LEVELS = np.linspace(0.05, 0.95, 20) # confidence levels for calibration

# ============================================================
#  SECTION 2 — REPRODUCIBILITY
# ============================================================

def set_seeds(seed):
    """Fix all random seeds for full reproducibility."""
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)


# ============================================================
#  SECTION 3 — MODEL BUILDER
# ============================================================

def build_model(input_dim, layers, dropout_rates, learning_rate=LEARNING_RATE):
    """
    Build a BNN-style neural network with MC Dropout.

    Dropout is applied after every hidden layer. During inference,
    dropout remains active (training=True), enabling Monte Carlo
    sampling to estimate predictive uncertainty.

    Parameters
    ----------
    input_dim    : number of input features
    layers       : list of hidden unit counts, e.g. [64, 32, 16]
    dropout_rates: list of dropout rates, one per hidden layer
    learning_rate: Adam learning rate

    Returns
    -------
    Compiled Keras model
    """
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.InputLayer(input_shape=(input_dim,)))

    for units, rate in zip(layers, dropout_rates):
        model.add(tf.keras.layers.Dense(
            units,
            activation="relu",
            kernel_initializer="he_normal"
        ))
        model.add(tf.keras.layers.Dropout(rate))

    # Single output neuron — predicts log(creep rupture life)
    model.add(tf.keras.layers.Dense(1, activation=None))

    model.compile(
        optimizer=Adam(
            learning_rate=learning_rate,
            beta_1=0.9, beta_2=0.999,
            epsilon=1e-7, clipnorm=1.0
        ),
        loss="mse",
        metrics=["mae"]
    )
    return model


# ============================================================
#  SECTION 4 — PREDICTION & METRICS
# ============================================================

def mc_predict(model, X, n_samples):
    """
    Monte Carlo Dropout inference.

    Runs the model n_samples times with dropout active (training=True).
    Returns the mean and std of the predictive distribution.

    Mean  → point prediction
    Std   → predictive uncertainty
    """
    samples = np.array([
        model(X, training=True).numpy().flatten()
        for _ in range(n_samples)
    ])
    return samples.mean(axis=0), samples.std(axis=0)


def nll_gaussian(y_true, mu, sigma, eps=1e-6):
    """
    Negative Log-Likelihood under a Gaussian predictive distribution.

    Lower is better. Jointly rewards accuracy and well-calibrated
    uncertainty — making it the ideal single criterion for model
    selection in a probabilistic setting.

    NLL = mean[ 0.5 * log(2π σ²) + 0.5 * (y - μ)² / σ² ]
    """
    sigma = np.maximum(sigma, eps)
    return np.mean(
        0.5 * np.log(2 * np.pi * sigma**2) +
        0.5 * ((y_true - mu)**2) / sigma**2
    )


def compute_ece(y_true, mu, sigma, n_bins=10):
    """
    Expected Calibration Error for regression.

    Measures the average gap between nominal and empirical coverage
    across confidence levels. Lower is better.
    """
    levels = np.linspace(0.05, 0.95, n_bins)
    empirical = []
    for c in levels:
        z = scipy.stats.norm.ppf((1 + c) / 2)
        covered = np.mean((y_true >= mu - z * sigma) & (y_true <= mu + z * sigma))
        empirical.append(covered)
    return float(np.mean(np.abs(np.array(empirical) - levels)))


def compute_coverage_and_iw(y_true, mu, sigma):
    """
    95% prediction interval coverage and mean interval width.

    Coverage: fraction of true values inside [μ ± 1.96σ]
    IW      : mean width of those intervals
    """
    lower = mu - 1.96 * sigma
    upper = mu + 1.96 * sigma
    coverage = float(np.mean((y_true >= lower) & (y_true <= upper)) * 100)
    iw       = float(np.mean(upper - lower))
    return coverage, iw


def compute_miscal_area (y_true, mu, sigma, levels=NOMINAL_LEVELS):
    """Compute miscalibration area (integral of |empirical - nominal|)."""
    empirical = []
    for c in levels:
        z = scipy.stats.norm.ppf((1 + c) / 2)
        covered = np.mean((y_true >= mu - z * sigma) & (y_true <= mu + z * sigma))
        empirical.append(covered)
    return float(np.trapz(np.abs(np.array(empirical) - levels), levels))


def get_callbacks(patience_stop, patience_lr):
    """Return standard training callbacks."""
    return [
        EarlyStopping(
            monitor="val_loss",
            patience=patience_stop,
            restore_best_weights=True,
            min_delta=1e-4,
            verbose=0
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=patience_lr,
            min_lr=1e-7,
            verbose=0
        )
    ]

# ============================================================
#  SECTION 5 — 5-FOLD CV ARCHITECTURE SELECTION
#  Procedure:
#    For each candidate architecture:
#      Split training data into 5 folds
#      For each fold:
#        Train on 4 folds → predict on held-out fold → compute NLL
#      Record mean NLL across 5 folds
#    Select the architecture with the lowest mean CV NLL
# ============================================================

def select_architecture_cv(X_train, y_train, seed):
    """
    Select the best architecture using 5-fold CV on training data.

    The test set is never used here — selection is entirely based
    on cross-validated NLL within the training set.

    Parameters
    ----------
    X_train : scaled training features  (numpy array)
    y_train : training targets          (numpy array)
    seed    : random seed for KFold shuffle

    Returns
    -------
    best_name   : name of the selected architecture
    best_params : dict with 'layers' and 'dropout' keys
    cv_summary  : DataFrame with NLL results for all architectures
    """
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    cv_records = []

    print(f"\n  5-Fold CV Architecture Selection (seed={seed})")
    print(f"  {'Architecture':<15} {'Fold NLLs':<45} {'Mean NLL':>10}")
    print(f"  {'-'*70}")

    for arch_name, arch_params in CANDIDATE_ARCHITECTURES.items():
        fold_nlls = []

        for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train)):

            # Split this fold
            X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
            y_fold_train, y_fold_val = y_train[train_idx], y_train[val_idx]

            # Each fold gets its own seed so results are reproducible
            fold_seed = seed + fold_idx
            set_seeds(fold_seed)

            # Build and train model on this fold
            model = build_model(
                input_dim=X_fold_train.shape[1],
                layers=arch_params["layers"],
                dropout_rates=arch_params["dropout"]
            )
            model.fit(
                X_fold_train, y_fold_train,
                validation_data=(X_fold_val, y_fold_val),
                epochs=EPOCHS_CV,
                batch_size=BATCH_SIZE,
                verbose=0,
                callbacks=get_callbacks(patience_stop=100, patience_lr=50)
            )

            # Predict on held-out fold
            mu_val, sig_val = mc_predict(model, X_fold_val, MC_SAMPLES_CV)
            fold_nll = nll_gaussian(y_fold_val, mu_val, sig_val)
            fold_nlls.append(fold_nll)

        mean_nll = np.mean(fold_nlls)
        std_nll  = np.std(fold_nlls)

        fold_str = "  ".join([f"{v:.3f}" for v in fold_nlls])
        print(f"  {arch_name:<15} [{fold_str}]   {mean_nll:>8.4f} ± {std_nll:.4f}")

        cv_records.append({
            "Architecture": arch_name,
            "Layers":       arch_params["layers"],
            "Dropout":      arch_params["dropout"],
            "Mean CV NLL":  round(mean_nll, 4),
            "Std CV NLL":   round(std_nll,  4),
        })

    cv_summary = pd.DataFrame(cv_records)

    # Select the architecture with the lowest mean CV NLL
    best_idx    = cv_summary["Mean CV NLL"].idxmin()
    best_name   = cv_summary.loc[best_idx, "Architecture"]
    best_params = {
        "layers":   cv_summary.loc[best_idx, "Layers"],
        "dropout":  cv_summary.loc[best_idx, "Dropout"]
    }

    print(f"\n  → Best architecture: {best_name} "
          f"(Mean CV NLL = {cv_summary.loc[best_idx, 'Mean CV NLL']:.4f})")

    return best_name, best_params, cv_summary


# ============================================================
#  SECTION 6 — MAIN EXPERIMENT LOOP
#
#  For each seed:
#    1. Split data into train (70%) and test (30%)
#       The test set is completely held out for final evaluation only.
#    2. Scale features (scaler fitted on train only — no leakage)
#    3. Select architecture via 5-fold CV on training data
#    4. Retrain selected architecture on full training set
#    5. Evaluate on test set — collect all metrics
#  After all seeds: compute mean ± std across seeds
# ============================================================

print("=" * 65)
print("  BNN MC DROPOUT — MULTI-SEED EVALUATION WITH 5-FOLD CV")
print("=" * 65)

all_results      = []   # one row per seed
all_cv_summaries = {}   # CV table per seed (for supplementary)
all_predictions  = []   # (y_test, mu_test, sig_test) per seed - for visualization

for seed in SEEDS:
    print(f"\n{'='*65}")
    print(f"  SEED {seed}")
    print(f"{'='*65}")

    # ----------------------------------------------------------
    #  Step 1: Data split
    #  70% train | 15% val (inside CV) | 15% test
    #  The val split here is not used directly — CV creates its
    #  own internal folds from the training set.
    #  We keep a hold-out test set that is never seen during
    #  architecture selection or training.
    # ----------------------------------------------------------
    set_seeds(seed)
    X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
        X, y, test_size=0.30, random_state=seed
    )

    # ----------------------------------------------------------
    #  Step 2: Feature scaling
    #  Scaler is fitted ONLY on training data.
    #  The same parameters are applied to the test set.
    # ----------------------------------------------------------
    scaler = MinMaxScaler()
    X_train_s = scaler.fit_transform(X_train_raw)
    X_test_s  = scaler.transform(X_test_raw)

    y_train = y_train_raw.values if hasattr(y_train_raw, "values") else np.array(y_train_raw)
    y_test  = y_test_raw.values  if hasattr(y_test_raw,  "values") else np.array(y_test_raw)

    # ----------------------------------------------------------
    #  Step 3: Architecture selection via 5-fold CV
    #  CV is performed entirely within X_train_s / y_train.
    #  Test set is untouched.
    # ----------------------------------------------------------
    best_name, best_params, cv_summary = select_architecture_cv(
        X_train_s, y_train, seed=seed
    )
    all_cv_summaries[seed] = cv_summary

    # ----------------------------------------------------------
    #  Step 4: Final model — retrain on full training set
    #  Now that the architecture is fixed, we use all training
    #  data. We use a small internal validation split (10%) only
    #  to drive early stopping — not for any metric reporting.
    # ----------------------------------------------------------
    print(f"\n  Training final model: {best_name}")

    # Small internal split for early stopping only
    X_tr, X_es, y_tr, y_es = train_test_split(
        X_train_s, y_train, test_size=0.10, random_state=seed
    )

    set_seeds(seed)
    final_model = build_model(
        input_dim=X_train_s.shape[1],
        layers=best_params["layers"],
        dropout_rates=best_params["dropout"]
    )

    history = final_model.fit(
        X_tr, y_tr,
        validation_data=(X_es, y_es),
        epochs=EPOCHS_FINAL,
        batch_size=BATCH_SIZE,
        verbose=0,
        callbacks=get_callbacks(patience_stop=150, patience_lr=50),
        shuffle=True
    )
    print(f"  Training stopped at epoch {len(history.history['loss'])}")

    # ----------------------------------------------------------
    #  Step 5: Test set evaluation (first and only time the test is used)
    #  500 MC samples for reliable uncertainty estimates.
    # ----------------------------------------------------------
    mu_test, sig_test = mc_predict(final_model, X_test_s, MC_SAMPLES)

    r2       = r2_score(y_test, mu_test)
    mae      = mean_absolute_error(y_test, mu_test)
    mse      = mean_squared_error(y_test, mu_test)
    nll      = nll_gaussian(y_test, mu_test, sig_test)
    sharp    = float(np.mean(sig_test))
    ece      = compute_ece(y_test, mu_test, sig_test)
    cov, iw  = compute_coverage_and_iw(y_test, mu_test, sig_test)
    miscal_area = compute_miscal_area(y_test, mu_test, sig_test)

    all_results.append({
        "Seed":         seed,
        "Architecture": best_name,
        "R²":           round(r2,   4),
        "MAE":          round(mae,  4),
        "MSE":          round(mse,  4),
        "NLL":          round(nll,  4),
        "Sharpness":    round(sharp,4),
        "ECE":          round(ece,  4),
        "Coverage (%)": round(cov,  2),
        "IW":           round(iw,   4),
        "Miscal. Area": round(miscal_area, 4),
    })
    #store predictions for visualization
    all_predictions.append({
        "seed":     seed,
        "y_test":   y_test,
        "mu_test":  mu_test,
        "sig_test": sig_test
    })

    print(f"\n  Test Results:")
    print(f"    R²={r2:.4f}  MAE={mae:.4f}  MSE={mse:.4f}")
    print(f"    NLL={nll:.4f}  Sharpness={sharp:.4f}  ECE={ece:.4f}")
    print(f"    Coverage={cov:.1f}%  IW={iw:.4f}")


# ============================================================
#  SECTION 7 — AGGREGATE RESULTS
#  This is what goes into our paper tables.
# ============================================================

print(f"\n{'='*65}")
print("  AGGREGATED RESULTS ACROSS ALL SEEDS")
print(f"{'='*65}")

METRIC_COLS  = ["R²", "MAE", "MSE", "NLL", "Sharpness", "ECE", "Coverage (%)", "IW", "Miscal. Area"]

results_df   = pd.DataFrame(all_results)
means = results_df[METRIC_COLS].mean().round(4)
stds  = results_df[METRIC_COLS].std().round(4)

# Per-seed table
print("\n📋 Per-seed results:")
print(results_df.to_string(index=False))

# Architecture selection frequency
arch_counts = results_df["Architecture"].value_counts()
print(f"\n📋 Architecture selection frequency across {len(SEEDS)} seeds:")
print(arch_counts.to_string())

# Paper-ready mean ± std table
print("\n📄 Paper-ready summary (BNN MCD — Mean ± Std across 10 seeds):")
paper_rows = {m: f"{means[m]:.3f} ± {stds[m]:.3f}" for m in METRIC_COLS}
paper_df = pd.DataFrame.from_dict(paper_rows, orient="index", columns=["BNN MCD"])
print(paper_df.to_string())


# ============================================================
#  SECTION 8 — VISUALISATIONS
# ============================================================

# --- Plot 1: Metric stability across seeds ---

# Set Elsevier-compatible font parameters
plt.rcParams.update({
    'font.size': 8,
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'axes.labelsize': 8,
    'axes.titlesize': 9,
    'legend.fontsize': 6,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'pdf.fonttype': 42,  # Editable text in PDF
    'ps.fonttype': 42,
})

fig, axes = plt.subplots(3, 3, figsize=(7.5, 6.5))
axes = axes.flatten()

for i, m in enumerate(METRIC_COLS):
    ax = axes[i]
    ax.bar(range(len(SEEDS)), results_df[m], color="steelblue", alpha=0.75,
           edgecolor="white")
    ax.axhline(means[m], color="crimson", linestyle="--", linewidth=1.5,
               label=f"Mean = {means[m]:.3f}")
    ax.fill_between([-0.5, len(SEEDS) - 0.5],
                    means[m] - stds[m], means[m] + stds[m],
                    alpha=0.15, color="crimson",
                    label=f"±1 SD = {stds[m]:.3f}")
    ax.set_title(m, fontweight="bold", fontsize=9, pad=5)
    ax.set_xlabel("Seed", fontsize=8)
    ax.set_ylabel("Value", fontsize=8)
    ax.set_xticks(range(len(SEEDS)))
    ax.set_xticklabels([str(s) for s in SEEDS], rotation=45, fontsize=7, ha='right')
    ax.tick_params(axis='both', labelsize=7)
    ax.grid(axis="y", alpha=0.25, linestyle='--', linewidth=0.5)

    # Add legend only to first subplot (saves space)
    if i == 0:
        ax.legend(loc='lower right', fontsize=6, frameon=True, 
                  fancybox=False, edgecolor='black', handlelength=1.5)


plt.suptitle("BNN MCD — Metric Stability Across 10 Random Seeds",
             fontsize=13, fontweight="bold", y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig("bnn_mcd_seed_stability.png", dpi=600, bbox_inches="tight")
plt.show()
print("\n✅ Saved: bnn_mcd_seed_stability.png")


# --- Plot 2: Calibration curves with +- std band ---
#
# Every calibration curve is evaluated at the same Nominal_Levels
# So emperical coverage values are directly comparable across seeds
# and can be averaged point-by-point.
# The shaded band shows how stable the calibration is across
# different data partitions - a norrow band means robust calibration.

all_empirical = []

for pred in all_predictions:
    empirical_this_seed = []
    for c in NOMINAL_LEVELS:
        z   =  scipy.stats.norm.ppf((1+c)/2)
        emp =  np.mean(
            (pred["y_test"] >= pred["mu_test"] - z * pred["sig_test"])&
            (pred["y_test"] <= pred["mu_test"] + z * pred["sig_test"])
        )
        empirical_this_seed.append(emp)
    all_empirical.append(empirical_this_seed)

all_empirical  = np.array(all_empirical)
mean_empirical = all_empirical.mean(axis=0)
std_empirical  = all_empirical.std (axis=0)
miscal_area    = float(np.trapz(
    np.abs(mean_empirical - NOMINAL_LEVELS), NOMINAL_LEVELS
))

fig, ax = plt.subplots(figsize=(3.5, 3.5))
ax.fill_between(NOMINAL_LEVELS,
                mean_empirical - std_empirical,
                mean_empirical + std_empirical,
                alpha = 0.20, color =  "steelblue", label =  f"± SD across {len(SEEDS)}seeds")
ax.plot(NOMINAL_LEVELS, mean_empirical, "o-", color="steelblue", linewidth = 1.5,  
        markersize = 3, label="BNN MCD (mean, n={len(SEEDS)} seeds)")
ax.plot([0, 1], [0, 1], "k--", linewidth = 1.0, label="Ideal calibration")
ax.set_xlabel("Nominal confidence level", fontsize = 9)
ax.set_ylabel("Empirical coverage", fontsize = 9)
ax.set_title("Calibration Curve — BNN MCD", fontsize = 10, fontweight = "bold")
ax.legend(fontsize = 7, loc='lower right')
ax.text (0.05, 0.88, 
         f"Miscalibration area = {miscal_area: 4f}",
         transform = ax.transAxes, fontsize = 8
        ,
        bbox = dict (boxstyle = "round, pad = 0.3", facecolor = "lightyellow", alpha = 0.8))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("bnn_mcd_calibration1.png", dpi=600, bbox_inches="tight")
plt.show()
print("✅ Saved: bnn_mcd_calibration1.png")


# --- Plot 3: Prediction intervals (last seed) ---
#
# Each seed has a different test set (different samples), so
# predictions cannot be averaged across seeds.
# We use the representative seed: the seed whose NLL is closest 
# to the mean NLL. This is the standard approach for showing a 
# "typical run" in ML papers. The title makes this explicit. 

mean_nll = results_df["NLL"].mean()
rep_idx  = int(np.argmin(np.abs(results_df["NLL"].values - mean_nll)))
rep_seed = int(results_df.iloc[rep_idx]["Seed"])
rep_nll  = results_df.iloc[rep_idx]["NLL"]
rep_cov  = results_df.iloc[rep_idx]["Coverage (%)"]

rep      = all_predictions [rep_idx]
y_rep    = rep ["y_test"]
mu_rep   = rep ["mu_test"]
sig_rep  = rep ["sig_test"] 

sort_idx = np.argsort(y_rep)

fig, ax  = plt.subplots(figsize = (7.5, 4))
ax.fill_between(range(len(y_rep)),
                mu_rep[sort_idx] - 1.96 * sig_rep[sort_idx],
                mu_rep[sort_idx] + 1.96 * sig_rep[sort_idx],
                alpha = 0.25, color = "steelblue", label = "95% Prediction Interval")
ax.plot(range(len(y_rep)), mu_rep [sort_idx],
        "o", markersize = 3, color = "steelblue", alpha = 0.7, label = "Predicted mean")
ax.plot(range(len(y_rep)), y_rep[sort_idx], 
        "r-", linewidth = 1.0, label = "Actual values")
ax.set_title(
    f"BNN MCD — Predictions with 95% Interval\n"
    f"Representative seed {rep_seed} "
    f"(NLL = {rep_nll:.3f} ≈ mean NLL = {mean_nll:.3f})"
    f"(Coverage = {rep_cov:.1f}%)",
    fontsize=9, fontweight = "bold"
)  
ax.legend(fontsize = 8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("bnn_mcd_predictions1.png", dpi=600, bbox_inches="tight")
plt.show()
print("✅ Saved: bnn_mcd_predictions1.png")

mcd_mean_empirical = mean_empirical.copy()
mcd_std_empirical  = std_empirical.copy()

mcd_all_predictions  = all_predictions.copy()
mcd_results_df       = results_df.copy()

# Save to disk
import joblib
joblib.dump(mcd_mean_empirical, 'mcd_mean_empirical.pkl')
joblib.dump(mcd_std_empirical, 'mcd_std_empirical.pkl')
joblib.dump(mcd_all_predictions, 'mcd_all_predictions.pkl')
joblib.dump(mcd_results_df, 'mcd_results_df.pkl')
print("✅ BNN MCD data saved to disk")

print(f"\n{'='*65}")
print("  BNN MCD COMPLETE")
print(f"{'='*65}")

#### Deep Ensemble 

In [ ]:
# ============================================================
#  DEEP ENSEMBLE (DE) — COMPLETE REVISED VERSION
#
#  Methodological improvements addressing Reviewer Point 3:
#  1. Architecture selected by 5-fold CV on training data
#     using mean NLL as the sole criterion (no arbitrary α)
#     → CV uses 1 member per fold (architecture selection only)
#       Full ensemble is built only during final evaluation
#  2. Repeated evaluation over 10 random seeds
#     → all metrics reported as mean ± std
#  3. Test set is never touched until final evaluation
#  4. All visualisations reflect full multi-seed evaluation
#
#  Prerequisites (run these notebook cells before this script):
#     import pandas as pd, numpy as np
#     uncreep = pd.read_excel("Combined - edited.xlsx")
#     uncreep["log_life"] = np.log(uncreep["Time to rupture/h"])
#     X = uncreep.drop(columns=["log_life", "Time to rupture/h"])
#     y = uncreep["log_life"]
# ============================================================

import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats

import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


# ============================================================
#  SECTION 1 — CONFIGURATION
#  All experiment settings live here.
# ============================================================

SEEDS = [42, 7, 13, 21, 37, 55, 71, 88, 99, 123]  # 10 seeds for repeated evaluation

# Candidate architectures (no dropout — DE uses deterministic members)
CANDIDATE_ARCHITECTURES = {
    "[24, 16, 8]":  {"layers": [24, 16, 8]},
    "[32, 16, 8]":  {"layers": [32, 16, 8]},
    "[48, 32, 16]": {"layers": [48, 32, 16]},
    "[64, 32, 16]": {"layers": [64, 32, 16]},
}

N_ENSEMBLE     = 5     # number of ensemble members for final evaluation
N_FOLDS        = 5     # folds for CV architecture selection
# NOTE: CV uses 1 member per fold (architecture selection only).
# The full N_ENSEMBLE members are trained only during final evaluation.
# This reduces CV compute from (N_FOLDS × N_ARCH × N_ENSEMBLE) to
# (N_FOLDS × N_ARCH × 1) — identical cost to BNN MCD's CV.

EPOCHS_CV      = 500   # max epochs per CV fold (1 member)
EPOCHS_FINAL   = 1000  # max epochs per final ensemble member
BATCH_SIZE     = 16
LEARNING_RATE  = 0.001
NOMINAL_LEVELS = np.linspace(0.05, 0.95, 20)  # confidence levels for calibration


# ============================================================
#  SECTION 2 — REPRODUCIBILITY
# ============================================================

def set_seeds(seed):
    """Fix NumPy, Python, and TensorFlow random states simultaneously."""
    np.random.seed(seed)
    random.seed(seed)
    tf.random.set_seed(seed)


# ============================================================
#  SECTION 3 — MODEL BUILDER
# ============================================================

def build_member(input_dim, layers, learning_rate=LEARNING_RATE, member_seed=None):
    """
    Build a single deterministic neural network — one ensemble member.

    Deep Ensembles derive uncertainty from diversity across independently
    trained members, each with a different random initialisation.
    No dropout is used: each member is a standard deterministic network.

    Parameters
    ----------
    input_dim    : number of input features
    layers       : list of hidden unit counts, e.g. [64, 32, 16]
    learning_rate: Adam learning rate
    member_seed  : seed for this member's weight initialisation
                   (different per member → diversity)

    Returns
    -------
    Compiled Keras model
    """
    if member_seed is not None:
        tf.random.set_seed(member_seed)

    model = tf.keras.Sequential()
    model.add(tf.keras.layers.InputLayer(input_shape=(input_dim,)))

    for units in layers:
        model.add(tf.keras.layers.Dense(
            units, activation="relu", kernel_initializer="he_normal"
        ))

    # Single output — predicts log(creep rupture life)
    # No dropout: ensemble diversity comes from different initialisations
    model.add(tf.keras.layers.Dense(1, activation=None))

    model.compile(
        optimizer=Adam(learning_rate=learning_rate,
                       beta_1=0.9, beta_2=0.999,
                       epsilon=1e-7, clipnorm=1.0),
        loss="mse", metrics=["mae"]
    )
    return model


# ============================================================
#  SECTION 4 — PREDICTION AND METRICS
# ============================================================

def ensemble_predict(members, X):
    """
    Deep Ensemble inference.

    Runs each member deterministically, then aggregates:
      Mean across members → point prediction
      Std  across members → predictive uncertainty (epistemic)

    Parameters
    ----------
    members : list of trained Keras models
    X       : input features (numpy array or tensor)

    Returns
    -------
    mu    : predictive mean  (n_samples,)
    sigma : predictive std   (n_samples,)
    """
    preds = np.array([m.predict(X, verbose=0).flatten() for m in members])
    return preds.mean(axis=0), preds.std(axis=0)


def nll_gaussian(y_true, mu, sigma, eps=1e-6):
    """
    Negative Log-Likelihood under a Gaussian predictive distribution.
    Lower is better. Jointly penalises inaccuracy and overconfidence.

    NLL = mean[ 0.5 * log(2π σ²)  +  0.5 * (y − μ)² / σ² ]
    """
    sigma = np.maximum(sigma, eps)
    return np.mean(
        0.5 * np.log(2 * np.pi * sigma**2) +
        0.5 * ((y_true - mu)**2) / sigma**2
    )


def compute_ece(y_true, mu, sigma, n_bins=10):
    """
    Expected Calibration Error for regression.
    Average gap between nominal and empirical coverage. Lower is better.
    """
    levels = np.linspace(0.05, 0.95, n_bins)
    empirical = []
    for c in levels:
        z = scipy.stats.norm.ppf((1 + c) / 2)
        covered = np.mean((y_true >= mu - z * sigma) & (y_true <= mu + z * sigma))
        empirical.append(covered)
    return float(np.mean(np.abs(np.array(empirical) - levels)))


def compute_coverage_and_iw(y_true, mu, sigma):
    """
    95% prediction interval coverage and mean interval width.
    Coverage: fraction of true values inside [μ ± 1.96σ]  (target ≈ 95%)
    IW      : mean width of those intervals (lower = sharper)
    """
    lower    = mu - 1.96 * sigma
    upper    = mu + 1.96 * sigma
    coverage = float(np.mean((y_true >= lower) & (y_true <= upper)) * 100)
    iw       = float(np.mean(upper - lower))
    return coverage, iw


def compute_miscal_area(y_true, mu, sigma, levels=NOMINAL_LEVELS):
    """Miscalibration area: integral of |empirical coverage − nominal level|."""
    empirical = []
    for c in levels:
        z = scipy.stats.norm.ppf((1 + c) / 2)
        covered = np.mean((y_true >= mu - z * sigma) & (y_true <= mu + z * sigma))
        empirical.append(covered)
    return float(np.trapz(np.abs(np.array(empirical) - levels), levels))


def get_callbacks(patience_stop, patience_lr):
    """Standard early stopping and learning rate reduction callbacks."""
    return [
        EarlyStopping(monitor="val_loss", patience=patience_stop,
                      restore_best_weights=True, min_delta=1e-4, verbose=0),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                          patience=patience_lr, min_lr=1e-7, verbose=0)
    ]


# ============================================================
#  SECTION 5 — 5-FOLD CV ARCHITECTURE SELECTION
#
#  KEY DESIGN DECISION:
#  Each CV fold trains ONE member (not the full ensemble).
#  Justification: we are selecting the architecture (layer
#  structure), not the ensemble size. A single member's NLL
#  is a reliable proxy for ensemble NLL, and this keeps CV
#  compute identical to BNN MCD (N_FOLDS × N_ARCH × 1 model).
#
#  The full N_ENSEMBLE members are only trained during the
#  final evaluation step (Section 6, Step 4).
#
#  Architecture selection is performed once on seed=42 and
#  the result is carried into all 10 evaluation seeds.
#  This is justified because ensemble diversity already comes
#  from different member initialisations within each seed —
#  repeating expensive CV per seed adds negligible information.
# ============================================================

def select_architecture_cv(X_train, y_train, seed):
    """
    Select the best architecture using 5-fold CV on training data.

    One member is trained per fold (for efficiency).
    The architecture with the lowest mean CV NLL is selected.
    The test set is never referenced here.

    Parameters
    ----------
    X_train : scaled training features  (numpy array)
    y_train : training targets          (numpy array)
    seed    : random seed for KFold shuffle

    Returns
    -------
    best_name   : name of the selected architecture
    best_params : dict with 'layers' key
    cv_summary  : DataFrame with CV NLL for all architectures
    """
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    cv_records = []

    print(f"\n  5-Fold CV Architecture Selection  (seed={seed})")
    print(f"  (1 member trained per fold — architecture selection only)")
    print(f"  {'Architecture':<15}  {'Fold NLLs':<50}  {'Mean NLL':>10}")
    print(f"  {'-'*80}")

    for arch_name, arch_params in CANDIDATE_ARCHITECTURES.items():
        fold_nlls = []

        for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train)):
            X_fold_tr, X_fold_val = X_train[train_idx], X_train[val_idx]
            y_fold_tr, y_fold_val = y_train[train_idx], y_train[val_idx]

            # Unique but reproducible seed per fold
            fold_seed = seed + fold_idx
            set_seeds(fold_seed)

            # Train ONE member for this fold
            member = build_member(
                input_dim=X_fold_tr.shape[1],
                layers=arch_params["layers"],
                member_seed=fold_seed
            )
            member.fit(
                X_fold_tr, y_fold_tr,
                validation_data=(X_fold_val, y_fold_val),
                epochs=EPOCHS_CV, batch_size=BATCH_SIZE, verbose=0,
                callbacks=get_callbacks(patience_stop=100, patience_lr=50)
            )

            # Evaluate NLL on held-out fold
            mu_val  = member.predict(X_fold_val, verbose=0).flatten()
            # Single deterministic member has no std — estimate from residuals
            # This is only used for architecture *ranking*, not final reporting
            residuals = y_fold_val - mu_val
            sigma_val = np.full_like(mu_val, np.std(residuals).clip(min=1e-6))
            fold_nlls.append(nll_gaussian(y_fold_val, mu_val, sigma_val))

        mean_nll = np.mean(fold_nlls)
        std_nll  = np.std(fold_nlls)
        fold_str = "  ".join([f"{v:.3f}" for v in fold_nlls])
        print(f"  {arch_name:<15}  [{fold_str}]  {mean_nll:>8.4f} ± {std_nll:.4f}")

        cv_records.append({
            "Architecture": arch_name,
            "Layers":       arch_params["layers"],
            "Mean CV NLL":  round(mean_nll, 4),
            "Std CV NLL":   round(std_nll,  4),
        })

    cv_summary = pd.DataFrame(cv_records)
    best_idx   = cv_summary["Mean CV NLL"].idxmin()
    best_name  = cv_summary.loc[best_idx, "Architecture"]
    best_params = {"layers": cv_summary.loc[best_idx, "Layers"]}

    print(f"\n  → Selected: {best_name}  "
          f"(Mean CV NLL = {cv_summary.loc[best_idx, 'Mean CV NLL']:.4f})")

    return best_name, best_params, cv_summary


# ============================================================
#  SECTION 6 — MAIN EXPERIMENT LOOP
#
#  Architecture selection (Section 5) is run ONCE on seed=42.
#  The selected architecture is then used for all 10 seeds.
#
#  For each seed:
#    Step 1 — Split: 70% train / 30% test  (test held out entirely)
#    Step 2 — Scale: MinMaxScaler fitted on train only (no leakage)
#    Step 3 — Train full ensemble (N_ENSEMBLE members) on training set
#             Each member gets a unique initialisation seed for diversity
#    Step 4 — Evaluate ensemble on test set, collect all metrics
#
#  After all seeds → report mean ± std
# ============================================================

print("=" * 65)
print("  DEEP ENSEMBLE — MULTI-SEED EVALUATION WITH 5-FOLD CV")
print("=" * 65)

# ── Step 0: Architecture selection (once, on seed=42) ───────
print("\n  Running architecture selection on seed=42 split...")

set_seeds(42)
X_cv_raw, _, y_cv_raw, _ = train_test_split(X, y, test_size=0.30, random_state=42)
scaler_cv   = MinMaxScaler()
X_cv_s      = scaler_cv.fit_transform(X_cv_raw)
y_cv        = (y_cv_raw.values if hasattr(y_cv_raw, "values")
               else np.array(y_cv_raw))

best_name, best_params, cv_summary = select_architecture_cv(X_cv_s, y_cv, seed=42)

print(f"\n  ✅ Architecture fixed for all seeds: {best_name}")
print(f"     Layers: {best_params['layers']}")
print(f"\n  CV Summary (supplementary material):")
print(cv_summary.to_string(index=False))

# ── Main seed loop ───────────────────────────────────────────
all_results     = []
all_predictions = []   # (y_test, mu_test, sig_test) per seed

for seed in SEEDS:
    print(f"\n{'='*65}")
    print(f"  SEED {seed}")
    print(f"{'='*65}")

    # ----------------------------------------------------------
    #  Step 1: Split
    #  70% train / 30% test. Test is held out until Step 4.
    # ----------------------------------------------------------
    set_seeds(seed)
    X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
        X, y, test_size=0.30, random_state=seed
    )

    # ----------------------------------------------------------
    #  Step 2: Scale
    #  Scaler fitted on training data only — no leakage.
    # ----------------------------------------------------------
    scaler    = MinMaxScaler()
    X_train_s = scaler.fit_transform(X_train_raw)
    X_test_s  = scaler.transform(X_test_raw)

    y_train = (y_train_raw.values if hasattr(y_train_raw, "values")
               else np.array(y_train_raw))
    y_test  = (y_test_raw.values  if hasattr(y_test_raw,  "values")
               else np.array(y_test_raw))

    # ----------------------------------------------------------
    #  Step 3: Train full ensemble on complete training set
    #
    #  Each member is initialised with a unique seed derived from
    #  the outer seed and the member index. This guarantees:
    #    - Different random initialisations → diversity within seed
    #    - Reproducible results across runs
    #
    #  A small 10% internal split drives early stopping only —
    #  it is not used for any reported metric.
    # ----------------------------------------------------------
    print(f"\n  Training {N_ENSEMBLE}-member ensemble: {best_name}")

    X_tr, X_es, y_tr, y_es = train_test_split(
        X_train_s, y_train, test_size=0.10, random_state=seed
    )

    ensemble_members = []

    for m_idx in range(N_ENSEMBLE):
        member_seed = seed * 100 + m_idx  # unique, reproducible per (seed, member)
        set_seeds(member_seed)

        member = build_member(
            input_dim=X_train_s.shape[1],
            layers=best_params["layers"],
            member_seed=member_seed
        )
        member.fit(
            X_tr, y_tr,
            validation_data=(X_es, y_es),
            epochs=EPOCHS_FINAL, batch_size=BATCH_SIZE, verbose=0,
            callbacks=get_callbacks(patience_stop=150, patience_lr=50),
            shuffle=True
        )
        ensemble_members.append(member)
        print(f"  Member {m_idx+1}/{N_ENSEMBLE} done.")

    # ----------------------------------------------------------
    #  Step 4: Test set evaluation
    #  First and only time the test set is used.
    # ----------------------------------------------------------
    mu_test, sig_test = ensemble_predict(ensemble_members, X_test_s)

    r2      = r2_score(y_test, mu_test)
    mae     = mean_absolute_error(y_test, mu_test)
    mse     = mean_squared_error(y_test, mu_test)
    nll     = nll_gaussian(y_test, mu_test, sig_test)
    sharp   = float(np.mean(sig_test))
    ece     = compute_ece(y_test, mu_test, sig_test)
    cov, iw = compute_coverage_and_iw(y_test, mu_test, sig_test)
    miscal  = compute_miscal_area(y_test, mu_test, sig_test)

    all_results.append({
        "Seed":         seed,
        "Architecture": best_name,
        "R²":           round(r2,    4),
        "MAE":          round(mae,   4),
        "MSE":          round(mse,   4),
        "NLL":          round(nll,   4),
        "Sharpness":    round(sharp, 4),
        "ECE":          round(ece,   4),
        "Coverage (%)": round(cov,   2),
        "IW":           round(iw,    4),
        "Miscal. Area": round(miscal,4),
    })

    all_predictions.append({
        "seed":    seed,
        "y_test":  y_test,
        "mu_test": mu_test,
        "sig_test": sig_test,
    })

    print(f"\n  Test Results:")
    print(f"    R²={r2:.4f}  MAE={mae:.4f}  MSE={mse:.4f}")
    print(f"    NLL={nll:.4f}  Sharpness={sharp:.4f}  ECE={ece:.4f}")
    print(f"    Coverage={cov:.1f}%  IW={iw:.4f}  Miscal.={miscal:.4f}")

    # Free memory before next seed
    del ensemble_members
    tf.keras.backend.clear_session()


# ============================================================
#  SECTION 7 — AGGREGATE RESULTS
#  These numbers go directly into your paper tables.
# ============================================================

METRIC_COLS = ["R²", "MAE", "MSE", "NLL", "Sharpness",
               "ECE", "Coverage (%)", "IW", "Miscal. Area"]

results_df = pd.DataFrame(all_results)
means      = results_df[METRIC_COLS].mean().round(4)
stds       = results_df[METRIC_COLS].std().round(4)

print(f"\n{'='*65}")
print("  AGGREGATED RESULTS ACROSS ALL SEEDS")
print(f"{'='*65}")

print("\n📋 Per-seed results:")
print(results_df.to_string(index=False))

print(f"\n📋 Architecture used across all {len(SEEDS)} seeds: {best_name}")

print("\n📄 Paper-ready summary — Deep Ensemble  (Mean ± Std, n=10 seeds):")
paper_rows = {m: f"{means[m]:.3f} ± {stds[m]:.3f}" for m in METRIC_COLS}
paper_df   = pd.DataFrame.from_dict(paper_rows, orient="index", columns=["DE"])
print(paper_df.to_string())


# ============================================================
#  SECTION 8 — VISUALISATIONS
#
#  Plot 1: Metric stability bars across seeds
#  Plot 2: Mean calibration curve with ±std band
#  Plot 3: Prediction intervals for representative seed
# ============================================================

plt.rcParams.update({
    "font.size": 8, "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "axes.labelsize": 8, "axes.titlesize": 9,
    "legend.fontsize": 6, "xtick.labelsize": 7,
    "ytick.labelsize": 7, "figure.dpi": 300,
    "savefig.dpi": 300, "pdf.fonttype": 42,
})


# ── Plot 1: Metric stability across seeds ───────────────────
fig, axes = plt.subplots(3, 3, figsize=(7.5, 6.5))
axes = axes.flatten()

for i, m in enumerate(METRIC_COLS):
    ax = axes[i]
    ax.bar(range(len(SEEDS)), results_df[m],
           color="darkorange", alpha=0.75, edgecolor="white")
    ax.axhline(means[m], color="crimson", linestyle="--", linewidth=1.5,
               label=f"Mean = {means[m]:.3f}")
    ax.fill_between([-0.5, len(SEEDS) - 0.5],
                    means[m] - stds[m], means[m] + stds[m],
                    alpha=0.15, color="crimson",
                    label=f"±1 SD = {stds[m]:.3f}")
    ax.set_title(m, fontweight="bold", fontsize=9, pad=5)
    ax.set_xlabel("Seed", fontsize=8)
    ax.set_ylabel("Value", fontsize=8)
    ax.set_xticks(range(len(SEEDS)))
    ax.set_xticklabels([str(s) for s in SEEDS], rotation=45, fontsize=7, ha="right")
    ax.grid(axis="y", alpha=0.25, linestyle="--", linewidth=0.5)
    if i == 0:
        ax.legend(loc="lower right", fontsize=6, frameon=True,
                  fancybox=False, edgecolor="black", handlelength=1.5)

plt.suptitle("Deep Ensemble — Metric Stability Across 10 Random Seeds",
             fontsize=11, fontweight="bold", y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("de_seed_stability.png", dpi=300, bbox_inches="tight")
plt.show()
print("\n✅ Saved: de_seed_stability.png")


# ── Plot 2: Mean calibration curve with ±std band ───────────
#
#  All curves share the same NOMINAL_LEVELS x-axis, so empirical
#  coverage values can be averaged point-by-point across seeds.
#  The ±std band shows calibration stability across data partitions.

all_empirical = []

for pred in all_predictions:
    empirical_this_seed = []
    for c in NOMINAL_LEVELS:
        z   = scipy.stats.norm.ppf((1 + c) / 2)
        emp = np.mean(
            (pred["y_test"] >= pred["mu_test"] - z * pred["sig_test"]) &
            (pred["y_test"] <= pred["mu_test"] + z * pred["sig_test"])
        )
        empirical_this_seed.append(emp)
    all_empirical.append(empirical_this_seed)

all_empirical  = np.array(all_empirical)          # shape: (n_seeds, n_levels)
mean_empirical = all_empirical.mean(axis=0)
std_empirical  = all_empirical.std(axis=0)
miscal_mean    = float(np.trapz(
    np.abs(mean_empirical - NOMINAL_LEVELS), NOMINAL_LEVELS
))

fig, ax = plt.subplots(figsize=(3.5, 3.5))
ax.fill_between(NOMINAL_LEVELS,
                mean_empirical - std_empirical,
                mean_empirical + std_empirical,
                alpha=0.20, color="darkorange",
                label=f"±1 SD across {len(SEEDS)} seeds")
ax.plot(NOMINAL_LEVELS, mean_empirical, "o-", color="darkorange",
        linewidth=1.5, markersize=3,
        label=f"DE (mean, n={len(SEEDS)} seeds)")
ax.plot([0, 1], [0, 1], "k--", linewidth=1.0, label="Ideal calibration")
ax.set_xlabel("Nominal confidence level", fontsize=9)
ax.set_ylabel("Empirical coverage", fontsize=9)
ax.set_title("Calibration Curve — Deep Ensemble", fontsize=10, fontweight="bold")
ax.legend(fontsize=7, loc="lower right")
ax.text(0.05, 0.88,
        f"Miscalibration area = {miscal_mean:.4f}",
        transform=ax.transAxes, fontsize=8,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("de_calibration.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Saved: de_calibration.png  "
      f"(mean miscalibration area = {miscal_mean:.4f})")


# ── Plot 3: Prediction intervals — representative seed ───────
#
#  Each seed has a different test set, so predictions cannot be
#  averaged. We use the representative seed: the seed whose NLL
#  is closest to the mean NLL across all seeds.

mean_nll  = results_df["NLL"].mean()
rep_idx   = int(np.argmin(np.abs(results_df["NLL"].values - mean_nll)))
rep_seed  = int(results_df.iloc[rep_idx]["Seed"])
rep_nll   = results_df.iloc[rep_idx]["NLL"]
rep_cov   = results_df.iloc[rep_idx]["Coverage (%)"]

rep       = all_predictions[rep_idx]
y_rep     = rep["y_test"]
mu_rep    = rep["mu_test"]
sig_rep   = rep["sig_test"]
sort_idx  = np.argsort(y_rep)

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.fill_between(range(len(y_rep)),
                mu_rep[sort_idx] - 1.96 * sig_rep[sort_idx],
                mu_rep[sort_idx] + 1.96 * sig_rep[sort_idx],
                alpha=0.25, color="darkorange", label="95% Prediction Interval")
ax.plot(range(len(y_rep)), mu_rep[sort_idx],
        "o", markersize=3, color="darkorange", alpha=0.7, label="Predicted mean")
ax.plot(range(len(y_rep)), y_rep[sort_idx],
        "r-", linewidth=1.0, label="Actual values")
ax.set_xlabel("Test samples (sorted by actual value)", fontsize=9)
ax.set_ylabel("Log rupture time", fontsize=9)
ax.set_title(
    f"Deep Ensemble — Predictions with 95% Intervals\n"
    f"Representative seed {rep_seed}  "
    f"(NLL = {rep_nll:.3f} ≈ mean NLL = {mean_nll:.3f},  "
    f"Coverage = {rep_cov:.1f}%)",
    fontsize=9, fontweight="bold"
)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("de_predictions.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Saved: de_predictions.png  (representative seed = {rep_seed})")

de_mean_empirical  = mean_empirical.copy()
de_std_empirical   = std_empirical.copy()

de_all_predictions   = all_predictions.copy()
de_results_df        = results_df.copy()

# Save to disk
import joblib
joblib.dump(de_mean_empirical, 'de_mean_empirical.pkl')
joblib.dump(de_std_empirical, 'de_std_empirical.pkl')
joblib.dump(de_all_predictions, 'de_all_predictions.pkl')
joblib.dump(de_results_df, 'de_results_df.pkl')
print("✅ DE data saved to disk")


print(f"\n{'='*65}")
print("  DEEP ENSEMBLE — COMPLETE")
print(f"{'='*65}")

### GPR 

In [ ]:
# ============================================================
#  GAUSSIAN PROCESS REGRESSION (GPR) — COMPLETE REVISED VERSION
#
#  Methodological improvements addressing Reviewer Point 3:
#  1. Kernel selected by 5-fold CV on training data using mean
#     NLL as the sole criterion (no arbitrary α)
#     → CV-NLL selects the kernel TYPE (structural decision)
#     → LML optimises kernel HYPERPARAMETERS within each type
#       (these are two distinct levels of model selection)
#  2. Repeated evaluation over 10 random seeds
#     → all metrics reported as mean ± std
#  3. Test set is never touched until final evaluation
#  4. All visualisations reflect full multi-seed evaluation
#
#  Prerequisites (run these notebook cells before this script):
#     import pandas as pd, numpy as np
#     uncreep = pd.read_excel("Combined - edited.xlsx")
#     uncreep["log_life"] = np.log(uncreep["Time to rupture/h"])
#     X = uncreep.drop(columns=["log_life", "Time to rupture/h"])
#     y = uncreep["log_life"]
# ============================================================

import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats
from scipy.spatial.distance import cdist

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF, Matern, RationalQuadratic, WhiteKernel, ConstantKernel as C
)
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


# ============================================================
#  SECTION 1 — CONFIGURATION
#  All experiment settings live here.
#
#  NOTE ON KERNEL CANDIDATES:
#  Each kernel is a function of input_dim because RBF_ARD uses
#  a per-feature length scale vector. The factory function
#  build_kernel() recreates kernels fresh each time they are
#  needed, avoiding state leakage between fits.
# ============================================================

SEEDS = [42, 7, 13, 21, 37, 55, 71, 88, 99, 123]

# Kernel candidate names — actual kernel objects are built fresh
# by build_kernel() to avoid state leakage between CV folds
CANDIDATE_KERNELS = [
    "RBF",
    "RBF_ARD",
    "Matern_2.5",
    "Matern_1.5",
    "RationalQuadratic",
]

N_FOLDS          = 5    # folds for CV kernel selection
N_RESTARTS_CV    = 5    # optimizer restarts during CV (faster)
N_RESTARTS_FINAL = 10   # optimizer restarts for final model (more thorough)
NOMINAL_LEVELS   = np.linspace(0.05, 0.95, 20)


# ============================================================
#  SECTION 2 — REPRODUCIBILITY
# ============================================================

def set_seeds(seed):
    """Fix NumPy and Python random states for reproducibility."""
    np.random.seed(seed)
    random.seed(seed)


# ============================================================
#  SECTION 3 — KERNEL FACTORY AND MODEL BUILDER
# ============================================================

def build_kernel(kernel_name, input_dim):
    """
    Create a fresh kernel object by name.

    Kernels must be recreated for each fit to avoid warm-starting
    from previous optimisation results, which would bias CV.
    All kernels include a WhiteKernel noise term for numerical
    stability and to model aleatoric uncertainty.

    Parameters
    ----------
    kernel_name : one of the CANDIDATE_KERNELS strings
    input_dim   : number of input features (needed for ARD)

    Returns
    -------
    sklearn kernel object
    """
    amplitude = C(1.0, (1e-3, 1e3))
    noise     = WhiteKernel(noise_level=1e-2,
                            noise_level_bounds=(1e-5, 1e1))

    if kernel_name == "RBF":
        # Isotropic RBF: single shared length scale
        return (amplitude
                * RBF(length_scale=1.0,
                      length_scale_bounds=(1e-2, 1e2))
                + noise)

    elif kernel_name == "RBF_ARD":
        # Automatic Relevance Determination: one length scale per feature
        # Allows the model to down-weight irrelevant input dimensions
        return (amplitude
                * RBF(length_scale=np.ones(input_dim),
                      length_scale_bounds=(1e-2, 1e2))
                + noise)

    elif kernel_name == "Matern_2.5":
        # Matérn ν=2.5: twice differentiable, often better for
        # physical data than infinitely smooth RBF
        return (amplitude
                * Matern(length_scale=1.0,
                         length_scale_bounds=(1e-2, 1e2), nu=2.5)
                + noise)

    elif kernel_name == "Matern_1.5":
        # Matérn ν=1.5: once differentiable, captures rougher functions
        return (amplitude
                * Matern(length_scale=1.0,
                         length_scale_bounds=(1e-2, 1e2), nu=1.5)
                + noise)

    elif kernel_name == "RationalQuadratic":
        # Rational Quadratic: equivalent to a scale mixture of RBFs
        # naturally captures multi-scale structure
        return (amplitude
                * RationalQuadratic(length_scale=1.0, alpha=1.0,
                                    length_scale_bounds=(1e-2, 1e2))
                + noise)

    else:
        raise ValueError(f"Unknown kernel: {kernel_name}")


def build_gpr(kernel_name, input_dim, n_restarts, seed):
    """
    Build a GaussianProcessRegressor with a fresh kernel.

    Parameters
    ----------
    kernel_name : kernel type string
    input_dim   : number of input features
    n_restarts  : optimizer restarts for LML maximisation
    seed        : random state for optimizer restarts

    Returns
    -------
    Unfitted GaussianProcessRegressor
    """
    return GaussianProcessRegressor(
        kernel=build_kernel(kernel_name, input_dim),
        alpha=1e-10,            # added to diagonal for numerical stability
        n_restarts_optimizer=n_restarts,
        normalize_y=True,       # normalise targets → better numerical conditioning
        random_state=seed
    )


# ============================================================
#  SECTION 4 — PREDICTION AND METRICS
# ============================================================

def gpr_predict(model, X):
    """
    GPR inference.

    GPR analytically returns the posterior mean and std —
    no sampling required. This is the key computational
    advantage of GPR over MC-based methods.

    Returns
    -------
    mu    : posterior predictive mean  (n_samples,)
    sigma : posterior predictive std   (n_samples,)
    """
    mu, sigma = model.predict(X, return_std=True)
    return mu.flatten(), sigma.flatten()


def nll_gaussian(y_true, mu, sigma, eps=1e-6):
    """
    Negative Log-Likelihood under a Gaussian predictive distribution.
    Lower is better. Jointly penalises inaccuracy and overconfidence.

    NLL = mean[ 0.5 * log(2π σ²)  +  0.5 * (y − μ)² / σ² ]
    """
    sigma = np.maximum(sigma, eps)
    return np.mean(
        0.5 * np.log(2 * np.pi * sigma**2) +
        0.5 * ((y_true - mu)**2) / sigma**2
    )


def compute_ece(y_true, mu, sigma, n_bins=10):
    """
    Expected Calibration Error for regression.
    Average gap between nominal and empirical coverage. Lower is better.
    """
    levels = np.linspace(0.05, 0.95, n_bins)
    empirical = []
    for c in levels:
        z = scipy.stats.norm.ppf((1 + c) / 2)
        covered = np.mean(
            (y_true >= mu - z * sigma) & (y_true <= mu + z * sigma)
        )
        empirical.append(covered)
    return float(np.mean(np.abs(np.array(empirical) - levels)))


def compute_coverage_and_iw(y_true, mu, sigma):
    """
    95% prediction interval coverage and mean interval width.
    Coverage: fraction of true values inside [μ ± 1.96σ]  (target ≈ 95%)
    IW      : mean width of those intervals (lower = sharper)
    """
    lower    = mu - 1.96 * sigma
    upper    = mu + 1.96 * sigma
    coverage = float(np.mean((y_true >= lower) & (y_true <= upper)) * 100)
    iw       = float(np.mean(upper - lower))
    return coverage, iw


def compute_miscal_area(y_true, mu, sigma, levels=NOMINAL_LEVELS):
    """Miscalibration area: integral of |empirical coverage − nominal level|."""
    empirical = []
    for c in levels:
        z = scipy.stats.norm.ppf((1 + c) / 2)
        covered = np.mean(
            (y_true >= mu - z * sigma) & (y_true <= mu + z * sigma)
        )
        empirical.append(covered)
    return float(np.trapz(np.abs(np.array(empirical) - levels), levels))


# ============================================================
#  SECTION 5 — 5-FOLD CV KERNEL SELECTION
#
#  TWO-LEVEL MODEL SELECTION FOR GPR:
#  ┌─ Outer level (this function) ──────────────────────────┐
#  │  CV-NLL selects the kernel TYPE                        │
#  │  (structural decision: RBF vs Matérn vs RQ etc.)       │
#  └────────────────────────────────────────────────────────┘
#  ┌─ Inner level (inside GPR.fit) ─────────────────────────┐
#  │  LML maximisation selects kernel HYPERPARAMETERS       │
#  │  (length scales, noise, amplitude — for a fixed type)  │
#  └────────────────────────────────────────────────────────┘
#
#  This distinction should be stated in the Methods section.
#
#  Kernel selection is performed once on seed=42 and carried
#  into all 10 evaluation seeds. This is justified because:
#  (a) kernel type selection is a structural decision unlikely
#      to change with different data splits;
#  (b) LML re-optimises hyperparameters for each seed's data,
#      so the model adapts fully to each split anyway.
# ============================================================

def select_kernel_cv(X_train, y_train, input_dim, seed):
    """
    Select the best kernel type using 5-fold CV on training data.

    For each candidate kernel:
      Train GPR on 4 folds → predict on held-out fold → compute NLL
      (LML optimises hyperparameters within each fold automatically)
    Select the kernel with the lowest mean CV NLL.

    The test set is never referenced here.

    Parameters
    ----------
    X_train   : scaled training features  (numpy array)
    y_train   : training targets          (numpy array)
    input_dim : number of input features  (for ARD kernel)
    seed      : random seed for KFold shuffle

    Returns
    -------
    best_kernel_name : name string of the selected kernel
    cv_summary       : DataFrame with CV NLL for all kernels
    """
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    cv_records = []

    print(f"\n  5-Fold CV Kernel Selection  (seed={seed})")
    print(f"  (LML optimises hyperparameters within each fold)")
    print(f"  {'Kernel':<20}  {'Fold NLLs':<52}  {'Mean NLL':>10}")
    print(f"  {'-'*87}")

    for kernel_name in CANDIDATE_KERNELS:
        fold_nlls = []

        for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X_train)):
            X_fold_tr,  X_fold_val  = X_train[train_idx], X_train[val_idx]
            y_fold_tr,  y_fold_val  = y_train[train_idx], y_train[val_idx]

            fold_seed = seed + fold_idx
            set_seeds(fold_seed)

            # Build fresh GPR — prevents warm-starting from previous fold
            gpr = build_gpr(
                kernel_name=kernel_name,
                input_dim=input_dim,
                n_restarts=N_RESTARTS_CV,
                seed=fold_seed
            )
            gpr.fit(X_fold_tr, y_fold_tr)

            # GPR returns mu and sigma analytically — no sampling needed
            mu_val, sig_val = gpr_predict(gpr, X_fold_val)
            fold_nlls.append(nll_gaussian(y_fold_val, mu_val, sig_val))

        mean_nll = np.mean(fold_nlls)
        std_nll  = np.std(fold_nlls)
        fold_str = "  ".join([f"{v:.3f}" for v in fold_nlls])
        print(f"  {kernel_name:<20}  [{fold_str}]  {mean_nll:>8.4f} ± {std_nll:.4f}")

        cv_records.append({
            "Kernel":       kernel_name,
            "Mean CV NLL":  round(mean_nll, 4),
            "Std CV NLL":   round(std_nll,  4),
        })

    cv_summary       = pd.DataFrame(cv_records)
    best_idx         = cv_summary["Mean CV NLL"].idxmin()
    best_kernel_name = cv_summary.loc[best_idx, "Kernel"]

    print(f"\n  → Selected: {best_kernel_name}  "
          f"(Mean CV NLL = {cv_summary.loc[best_idx, 'Mean CV NLL']:.4f})")

    return best_kernel_name, cv_summary


# ============================================================
#  SECTION 6 — MAIN EXPERIMENT LOOP
#
#  Kernel selection (Section 5) is run ONCE on seed=42.
#  The selected kernel TYPE is fixed for all 10 seeds.
#  LML re-optimises kernel HYPERPARAMETERS for each seed's data.
#
#  For each seed:
#    Step 1 — Split: 70% train / 30% test  (test held out entirely)
#    Step 2 — Scale: MinMaxScaler fitted on train only (no leakage)
#    Step 3 — Fit GPR on full training set with selected kernel
#             (LML maximisation tunes hyperparameters for this split)
#    Step 4 — Evaluate on test set, collect all metrics
#
#  After all seeds → report mean ± std
# ============================================================

print("=" * 65)
print("  GPR — MULTI-SEED EVALUATION WITH 5-FOLD CV")
print("=" * 65)

# ── Step 0: Kernel selection (once, on seed=42) ──────────────
print("\n  Running kernel selection on seed=42 split...")

set_seeds(42)
X_cv_raw, _, y_cv_raw, _ = train_test_split(X, y, test_size=0.30, random_state=42)
scaler_cv  = MinMaxScaler()
X_cv_s     = scaler_cv.fit_transform(X_cv_raw)
y_cv       = (y_cv_raw.values if hasattr(y_cv_raw, "values")
              else np.array(y_cv_raw))
input_dim  = X_cv_s.shape[1]

best_kernel_name, cv_summary = select_kernel_cv(X_cv_s, y_cv, input_dim, seed=42)

print(f"\n  ✅ Kernel fixed for all seeds: {best_kernel_name}")
print(f"\n  CV Summary (supplementary material):")
print(cv_summary.to_string(index=False))

# ── Main seed loop ───────────────────────────────────────────
all_results     = []
all_predictions = []   # (y_test, mu_test, sig_test) per seed

for seed in SEEDS:
    print(f"\n{'='*65}")
    print(f"  SEED {seed}")
    print(f"{'='*65}")

    # ----------------------------------------------------------
    #  Step 1: Split
    #  70% train / 30% test. Test is held out until Step 4.
    # ----------------------------------------------------------
    set_seeds(seed)
    X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
        X, y, test_size=0.30, random_state=seed
    )

    # ----------------------------------------------------------
    #  Step 2: Scale
    #  Scaler fitted on training data only — no leakage.
    # ----------------------------------------------------------
    scaler    = MinMaxScaler()
    X_train_s = scaler.fit_transform(X_train_raw)
    X_test_s  = scaler.transform(X_test_raw)

    y_train = (y_train_raw.values if hasattr(y_train_raw, "values")
               else np.array(y_train_raw))
    y_test  = (y_test_raw.values  if hasattr(y_test_raw,  "values")
               else np.array(y_test_raw))

    # ----------------------------------------------------------
    #  Step 3: Fit final GPR on full training set
    #
    #  A fresh kernel is built each seed to prevent warm-starting.
    #  LML optimisation (n_restarts=N_RESTARTS_FINAL) finds the
    #  best hyperparameters for this seed's training data.
    #  No validation split needed — GPR has no early stopping.
    # ----------------------------------------------------------
    print(f"\n  Fitting GPR ({best_kernel_name}) on training set...")
    set_seeds(seed)

    gpr = build_gpr(
        kernel_name=best_kernel_name,
        input_dim=input_dim,
        n_restarts=N_RESTARTS_FINAL,
        seed=seed
    )
    gpr.fit(X_train_s, y_train)

    print(f"  Optimised kernel: {gpr.kernel_}")
    print(f"  Log-marginal-likelihood: "
          f"{gpr.log_marginal_likelihood(gpr.kernel_.theta):.4f}")

    # ----------------------------------------------------------
    #  Step 4: Test set evaluation
    #  First and only time the test set is used.
    #  GPR returns mu and sigma analytically — no sampling.
    # ----------------------------------------------------------
    mu_test, sig_test = gpr_predict(gpr, X_test_s)

    r2      = r2_score(y_test, mu_test)
    mae     = mean_absolute_error(y_test, mu_test)
    mse     = mean_squared_error(y_test, mu_test)
    nll     = nll_gaussian(y_test, mu_test, sig_test)
    sharp   = float(np.mean(sig_test))
    ece     = compute_ece(y_test, mu_test, sig_test)
    cov, iw = compute_coverage_and_iw(y_test, mu_test, sig_test)
    miscal  = compute_miscal_area(y_test, mu_test, sig_test)

    all_results.append({
        "Seed":         seed,
        "Kernel":       best_kernel_name,
        "R²":           round(r2,    4),
        "MAE":          round(mae,   4),
        "MSE":          round(mse,   4),
        "NLL":          round(nll,   4),
        "Sharpness":    round(sharp, 4),
        "ECE":          round(ece,   4),
        "Coverage (%)": round(cov,   2),
        "IW":           round(iw,    4),
        "Miscal. Area": round(miscal,4),
    })

    all_predictions.append({
        "seed":     seed,
        "y_test":   y_test,
        "mu_test":  mu_test,
        "sig_test": sig_test,
        "X_test_s": X_test_s,   # stored for uncertainty–distance plot
        "X_train_s": X_train_s, # stored for uncertainty–distance plot
    })

    print(f"\n  Test Results:")
    print(f"    R²={r2:.4f}  MAE={mae:.4f}  MSE={mse:.4f}")
    print(f"    NLL={nll:.4f}  Sharpness={sharp:.4f}  ECE={ece:.4f}")
    print(f"    Coverage={cov:.1f}%  IW={iw:.4f}  Miscal.={miscal:.4f}")


# ============================================================
#  SECTION 7 — AGGREGATE RESULTS
#  These numbers go directly into your paper tables.
# ============================================================

METRIC_COLS = ["R²", "MAE", "MSE", "NLL", "Sharpness",
               "ECE", "Coverage (%)", "IW", "Miscal. Area"]

results_df = pd.DataFrame(all_results)
means      = results_df[METRIC_COLS].mean().round(4)
stds       = results_df[METRIC_COLS].std().round(4)

print(f"\n{'='*65}")
print("  AGGREGATED RESULTS ACROSS ALL SEEDS")
print(f"{'='*65}")

print("\n📋 Per-seed results:")
print(results_df.to_string(index=False))

print(f"\n📋 Kernel used across all {len(SEEDS)} seeds: {best_kernel_name}")

print("\n📄 Paper-ready summary — GPR  (Mean ± Std, n=10 seeds):")
paper_rows = {m: f"{means[m]:.3f} ± {stds[m]:.3f}" for m in METRIC_COLS}
paper_df   = pd.DataFrame.from_dict(paper_rows, orient="index", columns=["GPR"])
print(paper_df.to_string())


# ============================================================
#  SECTION 8 — VISUALISATIONS
#
#  Plot 1: Metric stability bars across seeds
#  Plot 2: Mean calibration curve with ±std band
#  Plot 3: Prediction intervals for representative seed
#  Plot 4: Uncertainty vs distance from training data
#          (GPR-specific physical insight — kept from original)
# ============================================================

plt.rcParams.update({
    "font.size": 8, "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "axes.labelsize": 8, "axes.titlesize": 9,
    "legend.fontsize": 6, "xtick.labelsize": 7,
    "ytick.labelsize": 7, "figure.dpi": 300,
    "savefig.dpi": 300, "pdf.fonttype": 42,
})


# ── Plot 1: Metric stability across seeds ───────────────────
fig, axes = plt.subplots(3, 3, figsize=(7.5, 6.5))
axes = axes.flatten()

for i, m in enumerate(METRIC_COLS):
    ax = axes[i]
    ax.bar(range(len(SEEDS)), results_df[m],
           color="seagreen", alpha=0.75, edgecolor="white")
    ax.axhline(means[m], color="crimson", linestyle="--", linewidth=1.5,
               label=f"Mean = {means[m]:.3f}")
    ax.fill_between([-0.5, len(SEEDS) - 0.5],
                    means[m] - stds[m], means[m] + stds[m],
                    alpha=0.15, color="crimson",
                    label=f"±1 SD = {stds[m]:.3f}")
    ax.set_title(m, fontweight="bold", fontsize=9, pad=5)
    ax.set_xlabel("Seed", fontsize=8)
    ax.set_ylabel("Value", fontsize=8)
    ax.set_xticks(range(len(SEEDS)))
    ax.set_xticklabels([str(s) for s in SEEDS], rotation=45, fontsize=7, ha="right")
    ax.grid(axis="y", alpha=0.25, linestyle="--", linewidth=0.5)
    if i == 0:
        ax.legend(loc="lower right", fontsize=6, frameon=True,
                  fancybox=False, edgecolor="black", handlelength=1.5)

plt.suptitle("GPR — Metric Stability Across 10 Random Seeds",
             fontsize=11, fontweight="bold", y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("gpr_seed_stability.png", dpi=300, bbox_inches="tight")
plt.show()
print("\n✅ Saved: gpr_seed_stability.png")


# ── Plot 2: Mean calibration curve with ±std band ───────────
#
#  All curves share the same NOMINAL_LEVELS x-axis, so empirical
#  coverage values can be averaged point-by-point across seeds.
#  The ±std band shows calibration stability across data splits.

all_empirical = []

for pred in all_predictions:
    empirical_this_seed = []
    for c in NOMINAL_LEVELS:
        z   = scipy.stats.norm.ppf((1 + c) / 2)
        emp = np.mean(
            (pred["y_test"] >= pred["mu_test"] - z * pred["sig_test"]) &
            (pred["y_test"] <= pred["mu_test"] + z * pred["sig_test"])
        )
        empirical_this_seed.append(emp)
    all_empirical.append(empirical_this_seed)

all_empirical  = np.array(all_empirical)          # shape: (n_seeds, n_levels)
mean_empirical = all_empirical.mean(axis=0)
std_empirical  = all_empirical.std(axis=0)
miscal_mean    = float(np.trapz(
    np.abs(mean_empirical - NOMINAL_LEVELS), NOMINAL_LEVELS
))

fig, ax = plt.subplots(figsize=(3.5, 3.5))
ax.fill_between(NOMINAL_LEVELS,
                mean_empirical - std_empirical,
                mean_empirical + std_empirical,
                alpha=0.20, color="seagreen",
                label=f"±1 SD across {len(SEEDS)} seeds")
ax.plot(NOMINAL_LEVELS, mean_empirical, "o-", color="seagreen",
        linewidth=1.5, markersize=3,
        label=f"GPR (mean, n={len(SEEDS)} seeds)")
ax.plot([0, 1], [0, 1], "k--", linewidth=1.0, label="Ideal calibration")
ax.set_xlabel("Nominal confidence level", fontsize=9)
ax.set_ylabel("Empirical coverage", fontsize=9)
ax.set_title("Calibration Curve — GPR", fontsize=10, fontweight="bold")
ax.legend(fontsize=7, loc="lower right")
ax.text(0.05, 0.88,
        f"Miscalibration area = {miscal_mean:.4f}",
        transform=ax.transAxes, fontsize=8,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("gpr_calibration.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Saved: gpr_calibration.png  "
      f"(mean miscalibration area = {miscal_mean:.4f})")


# ── Plot 3: Prediction intervals — representative seed ───────
#
#  Each seed has a different test set, so predictions cannot be
#  averaged. We use the representative seed: the seed whose NLL
#  is closest to the mean NLL across all seeds.

mean_nll  = results_df["NLL"].mean()
rep_idx   = int(np.argmin(np.abs(results_df["NLL"].values - mean_nll)))
rep_seed  = int(results_df.iloc[rep_idx]["Seed"])
rep_nll   = results_df.iloc[rep_idx]["NLL"]
rep_cov   = results_df.iloc[rep_idx]["Coverage (%)"]

rep       = all_predictions[rep_idx]
y_rep     = rep["y_test"]
mu_rep    = rep["mu_test"]
sig_rep   = rep["sig_test"]
sort_idx  = np.argsort(y_rep)

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.fill_between(range(len(y_rep)),
                mu_rep[sort_idx] - 1.96 * sig_rep[sort_idx],
                mu_rep[sort_idx] + 1.96 * sig_rep[sort_idx],
                alpha=0.25, color="seagreen", label="95% Prediction Interval")
ax.plot(range(len(y_rep)), mu_rep[sort_idx],
        "o", markersize=3, color="seagreen", alpha=0.7, label="Predicted mean")
ax.plot(range(len(y_rep)), y_rep[sort_idx],
        "r-", linewidth=1.0, label="Actual values")
ax.set_xlabel("Test samples (sorted by actual value)", fontsize=9)
ax.set_ylabel("Log rupture time", fontsize=9)
ax.set_title(
    f"GPR — Predictions with 95% Intervals\n"
    f"Representative seed {rep_seed}  "
    f"(NLL = {rep_nll:.3f} ≈ mean NLL = {mean_nll:.3f},  "
    f"Coverage = {rep_cov:.1f}%)",
    fontsize=9, fontweight="bold"
)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("gpr_predictions.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Saved: gpr_predictions.png  (representative seed = {rep_seed})")


# ── Plot 4: Uncertainty vs distance from training data ───────
#
#  This is a GPR-specific physical insight kept from the original.
#  GPR uncertainty should increase as test points move further
#  from the training distribution — this is the epistemic
#  uncertainty the kernel structure encodes.
#
#  Using the representative seed for consistency with Plot 3.
#  Correlation values are reported across all seeds for robustness.

print("\n  Computing uncertainty–distance analysis...")

dist_corr_uncertainty = []
dist_corr_error       = []

for pred in all_predictions:
    dists     = cdist(pred["X_test_s"], pred["X_train_s"],
                      metric="euclidean").min(axis=1)
    errors    = np.abs(pred["y_test"] - pred["mu_test"])
    corr_unc  = np.corrcoef(dists, pred["sig_test"])[0, 1]
    corr_err  = np.corrcoef(dists, errors)[0, 1]
    dist_corr_uncertainty.append(corr_unc)
    dist_corr_error.append(corr_err)

mean_corr_unc = np.mean(dist_corr_uncertainty)
std_corr_unc  = np.std(dist_corr_uncertainty)
mean_corr_err = np.mean(dist_corr_error)
std_corr_err  = np.std(dist_corr_error)

print(f"\n  Uncertainty–Distance Correlation (mean ± std across seeds):")
print(f"    Distance vs Uncertainty : {mean_corr_unc:.4f} ± {std_corr_unc:.4f}")
print(f"    Distance vs Error       : {mean_corr_err:.4f} ± {std_corr_err:.4f}")

# Plot for the representative seed
rep_dists  = cdist(rep["X_test_s"], rep["X_train_s"],
                   metric="euclidean").min(axis=1)
rep_errors = np.abs(rep["y_test"] - rep["mu_test"])

fig, axes = plt.subplots(1, 2, figsize=(7.5, 3.5))

axes[0].scatter(rep_dists, rep["sig_test"],
                alpha=0.5, s=15, color="seagreen")
axes[0].set_xlabel("Min distance to training data", fontsize=9)
axes[0].set_ylabel("Predictive uncertainty (std)", fontsize=9)
axes[0].set_title(
    f"Uncertainty vs Distance\n"
    f"r = {mean_corr_unc:.3f} ± {std_corr_unc:.3f} (across 10 seeds)",
    fontsize=9, fontweight="bold"
)
axes[0].grid(alpha=0.3)

axes[1].scatter(rep_dists, rep_errors,
                alpha=0.5, s=15, color="seagreen")
axes[1].set_xlabel("Min distance to training data", fontsize=9)
axes[1].set_ylabel("Absolute prediction error", fontsize=9)
axes[1].set_title(
    f"Error vs Distance\n"
    f"r = {mean_corr_err:.3f} ± {std_corr_err:.3f} (across 10 seeds)",
    fontsize=9, fontweight="bold"
)
axes[1].grid(alpha=0.3)

plt.suptitle(
    f"GPR — Uncertainty and Error vs Distance from Training Data\n"
    f"(shown for representative seed {rep_seed}; "
    f"correlations reported across all seeds)",
    fontsize=9, fontweight="bold"
)
plt.tight_layout()
plt.savefig("gpr_uncertainty_distance.png", dpi=300, bbox_inches="tight")
plt.show()
print("✅ Saved: gpr_uncertainty_distance.png")

gpr_mean_empirical = mean_empirical.copy()
gpr_std_empirical  = std_empirical.copy()

gpr_all_predictions  = all_predictions.copy()
gpr_results_df       = results_df.copy()

# Save to disk
import joblib
joblib.dump(gpr_mean_empirical, 'gpr_mean_empirical.pkl')
joblib.dump(gpr_std_empirical, 'gpr_std_empirical.pkl')
joblib.dump(gpr_all_predictions, 'gpr_all_predictions.pkl')
joblib.dump(gpr_results_df, 'gpr_results_df.pkl')
print("✅ GPR data saved to disk")


print(f"\n{'='*65}")
print("  GPR — COMPLETE")
print(f"{'='*65}")

### NGBOOST

In [ ]:
# ============================================================
#  NATURAL GRADIENT BOOSTING (NGBoost) — COMPLETE VERSION
#
#  Methodology consistent with all four revised models:
#  1. Hyperparameter selection by 5-fold CV on training data
#     using mean NLL as the sole criterion (no arbitrary α)
#     → CV run once on seed=42, carried into all 10 seeds
#  2. Repeated evaluation over 10 random seeds
#     → all metrics reported as mean ± std
#  3. Test set never touched until final evaluation
#  4. All visualisations reflect full multi-seed evaluation
#
#  Installation (run once before this script):
#     pip install ngboost
#
#  NGBoost key concept:
#  Unlike the other four models, NGBoost treats the output as
#  a probability distribution and fits the distribution
#  parameters (mean μ and variance σ²) directly via natural
#  gradient boosting on decision tree base learners.
#  This provides a native predictive distribution without
#  requiring sampling, ensembling, or kernel methods.
#  The predictive distribution is Gaussian by default, making
#  NLL directly comparable across all five models.
#
#  Prerequisites (run these notebook cells before this script):
#     import pandas as pd, numpy as np
#     uncreep = pd.read_excel("Combined - edited.xlsx")
#     uncreep["log_life"] = np.log(uncreep["Time to rupture/h"])
#     X = uncreep.drop(columns=["log_life", "Time to rupture/h"])
#     y = uncreep["log_life"]
# ============================================================

import numpy as np
import random
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats

from ngboost import NGBRegressor
from ngboost.distns import Normal
from ngboost.scores import LogScore

from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


# ============================================================
#  SECTION 1 — CONFIGURATION

#  NGBoost has four main hyperparameters to tune:
#
#  n_estimators : number of boosting iterations
#                 more = more expressive, higher risk of overfitting
#  learning_rate: shrinkage factor applied to each tree's contribution
#                 smaller = slower learning, often better generalisation
#  max_depth    : depth of each base decision tree learner
#                 shallow trees (1-3) are standard for boosting
#  minibatch_frac: fraction of training data used per iteration
#                 < 1.0 introduces stochasticity, improves generalisation
#
#  The combination of n_estimators and learning_rate is the most
#  important: low LR + many estimators often outperforms high LR + few.
# ============================================================

SEEDS = [42, 7, 13, 21, 37, 55, 71, 88, 99, 123]

# Candidate hyperparameter configurations for CV selection
# Each entry is one complete configuration to evaluate
CANDIDATE_CONFIGS = {
    "n200_lr0.05_d2": {
        "n_estimators":   200,
        "learning_rate":  0.05,
        "max_depth":      2,
        "minibatch_frac": 1.0,
    },
    "n200_lr0.05_d3": {
        "n_estimators":   200,
        "learning_rate":  0.05,
        "max_depth":      3,
        "minibatch_frac": 1.0,
    },
    "n500_lr0.02_d2": {
        "n_estimators":   500,
        "learning_rate":  0.02,
        "max_depth":      2,
        "minibatch_frac": 1.0,
    },
    "n500_lr0.02_d3": {
        "n_estimators":   500,
        "learning_rate":  0.02,
        "max_depth":      3,
        "minibatch_frac": 1.0,
    },
    "n500_lr0.05_d2": {
        "n_estimators":   500,
        "learning_rate":  0.05,
        "max_depth":      2,
        "minibatch_frac": 0.8,
    },
}

N_FOLDS        = 5
NOMINAL_LEVELS = np.linspace(0.05, 0.95, 20)


# ============================================================
#  SECTION 2 — REPRODUCIBILITY
# ============================================================

def set_seeds(seed):
    """Fix NumPy and Python random states for reproducibility."""
    np.random.seed(seed)
    random.seed(seed)


# ============================================================
#  SECTION 3 — MODEL BUILDER
# ============================================================

def build_ngboost(config, seed):
    """
    Build an NGBRegressor with a given hyperparameter configuration.

    NGBoost uses:
      - Normal distribution as the output distribution
        → directly comparable NLL with all other models
      - LogScore (negative log-likelihood) as the scoring rule
        → matches the NLL criterion used for CV selection
      - DecisionTreeRegressor as the base learner
        → standard choice; max_depth controls expressiveness

    Parameters
    ----------
    config : dict with n_estimators, learning_rate, max_depth,
             minibatch_frac
    seed   : random state for reproducibility

    Returns
    -------
    Unfitted NGBRegressor
    """
    base_learner = DecisionTreeRegressor(
        max_depth=config["max_depth"],
        random_state=seed
    )

    return NGBRegressor(
        Dist=Normal,
        Score=LogScore,
        Base=base_learner,
        n_estimators=config["n_estimators"],
        learning_rate=config["learning_rate"],
        minibatch_frac=config["minibatch_frac"],
        random_state=seed,
        verbose=False
    )


# ============================================================
#  SECTION 4 — PREDICTION AND METRICS
#
#  NGBoost prediction is fundamentally different from the other
#  four models: it returns a distribution object directly.
#  From the distribution object we extract:
#    dist.loc   → predictive mean μ
#    dist.scale → predictive std σ
#  No sampling, no forward passes, no kernel inversion.
# ============================================================

def ngb_predict(model, X):
    """
    NGBoost inference.

    Returns the predictive mean and std from the fitted
    Normal distribution at each test point.

    Parameters
    ----------
    model : fitted NGBRegressor
    X     : input features (numpy array)

    Returns
    -------
    mu    : predictive mean  (n_samples,)
    sigma : predictive std   (n_samples,)
    """
    dist  = model.pred_dist(X)
    mu    = dist.loc                           # mean of Normal distribution
    sigma = np.maximum(dist.scale, 1e-6)      # std, clipped for numerical safety
    return mu, sigma


def nll_gaussian(y_true, mu, sigma, eps=1e-6):
    """
    Gaussian NLL. Lower is better.

    NLL = mean[ 0.5 log(2π σ²)  +  0.5 (y − μ)² / σ² ]

    Note: NGBoost is trained to minimise exactly this quantity
    (via LogScore), so NLL is the natural evaluation metric
    and is directly comparable across all five models.
    """
    sigma = np.maximum(sigma, eps)
    return np.mean(
        0.5 * np.log(2 * np.pi * sigma**2) +
        0.5 * ((y_true - mu)**2) / sigma**2
    )


def compute_ece(y_true, mu, sigma, n_bins=10):
    """
    Expected Calibration Error for regression.
    Average gap between nominal and empirical coverage.
    Lower is better.
    """
    levels    = np.linspace(0.05, 0.95, n_bins)
    empirical = []
    for c in levels:
        z = scipy.stats.norm.ppf((1 + c) / 2)
        covered = np.mean(
            (y_true >= mu - z * sigma) & (y_true <= mu + z * sigma)
        )
        empirical.append(covered)
    return float(np.mean(np.abs(np.array(empirical) - levels)))


def compute_coverage_and_iw(y_true, mu, sigma):
    """
    95% prediction interval coverage and mean interval width.
    Coverage: fraction of true values inside [μ ± 1.96σ]
    IW      : mean width of those intervals
    """
    lower    = mu - 1.96 * sigma
    upper    = mu + 1.96 * sigma
    coverage = float(np.mean((y_true >= lower) & (y_true <= upper)) * 100)
    iw       = float(np.mean(upper - lower))
    return coverage, iw


def compute_miscal_area(y_true, mu, sigma, levels=NOMINAL_LEVELS):
    """Miscalibration area: integral of |empirical coverage − nominal|."""
    empirical = []
    for c in levels:
        z = scipy.stats.norm.ppf((1 + c) / 2)
        covered = np.mean(
            (y_true >= mu - z * sigma) & (y_true <= mu + z * sigma)
        )
        empirical.append(covered)
    return float(np.trapz(np.abs(np.array(empirical) - levels), levels))


# ============================================================
#  SECTION 5 — 5-FOLD CV HYPERPARAMETER SELECTION
#
#  For NGBoost, CV tunes the full hyperparameter configuration
#  (n_estimators, learning_rate, max_depth, minibatch_frac)
#  simultaneously — not just the architecture depth.
#  This is analogous to kernel selection in GPR.
#
#  Selection is performed once on seed=42 and carried into
#  all 10 evaluation seeds. Justified because hyperparameter
#  selection is a structural decision unlikely to depend on the
#  specific data split, and NGBoost fitting is fast enough that
#  this CV is not computationally limiting.
#
#  Early stopping is used within each CV fold to prevent
#  overfitting and reduce computation: training stops when
#  the held-out fold NLL does not improve for 50 rounds.
# ============================================================

def select_config_cv(X_train, y_train, seed):
    """
    Select the best hyperparameter configuration by 5-fold CV.

    For each candidate configuration:
      Train NGBoost on 4 folds → predict on held-out fold → NLL
    Select the configuration with the lowest mean CV NLL.

    Early stopping is applied within each fold using the
    held-out fold as the validation set — this correctly uses
    the fold boundary and does not leak test information.

    Parameters
    ----------
    X_train : scaled training features  (numpy array)
    y_train : training targets          (numpy array)
    seed    : random seed for KFold shuffle

    Returns
    -------
    best_config_name : string key into CANDIDATE_CONFIGS
    best_config      : dict with hyperparameter values
    cv_summary       : DataFrame with CV NLL for all configs
    """
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    cv_records = []

    print(f"\n  5-Fold CV Hyperparameter Selection  (seed={seed})")
    print(f"  {'Config':<22}  {'Fold NLLs':<52}  {'Mean NLL':>10}")
    print(f"  {'-'*88}")

    for config_name, config in CANDIDATE_CONFIGS.items():
        fold_nlls = []

        for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(X_train)):
            X_fold_tr,  X_fold_val  = X_train[tr_idx],  X_train[val_idx]
            y_fold_tr,  y_fold_val  = y_train[tr_idx],  y_train[val_idx]

            fold_seed = seed + fold_idx
            set_seeds(fold_seed)

            model = build_ngboost(config, seed=fold_seed)

            # Early stopping: use the fold val set as the ES monitor.
            # X_val and Y_val here are the fold validation arrays —
            # not the outer test set. This is methodologically correct.
            model.fit(
                X_fold_tr, y_fold_tr,
                X_val=X_fold_val,
                Y_val=y_fold_val,
                early_stopping_rounds=50
            )

            mu_val, sig_val = ngb_predict(model, X_fold_val)
            fold_nlls.append(nll_gaussian(y_fold_val, mu_val, sig_val))

        mean_nll = np.mean(fold_nlls)
        std_nll  = np.std(fold_nlls)
        fold_str = "  ".join([f"{v:.3f}" for v in fold_nlls])
        print(f"  {config_name:<22}  [{fold_str}]  {mean_nll:>8.4f} ± {std_nll:.4f}")

        cv_records.append({
            "Config":       config_name,
            "n_estimators": config["n_estimators"],
            "learning_rate":config["learning_rate"],
            "max_depth":    config["max_depth"],
            "minibatch_frac":config["minibatch_frac"],
            "Mean CV NLL":  round(mean_nll, 4),
            "Std CV NLL":   round(std_nll,  4),
        })

    cv_summary       = pd.DataFrame(cv_records)
    best_idx         = cv_summary["Mean CV NLL"].idxmin()
    best_config_name = cv_summary.loc[best_idx, "Config"]
    best_config      = CANDIDATE_CONFIGS[best_config_name]

    print(f"\n  → Selected: {best_config_name}  "
          f"(Mean CV NLL = {cv_summary.loc[best_idx, 'Mean CV NLL']:.4f})")
    print(f"     n_estimators={best_config['n_estimators']}  "
          f"learning_rate={best_config['learning_rate']}  "
          f"max_depth={best_config['max_depth']}  "
          f"minibatch_frac={best_config['minibatch_frac']}")

    return best_config_name, best_config, cv_summary


# ============================================================
#  SECTION 6 — MAIN EXPERIMENT LOOP
#
#  Hyperparameter selection (Section 5) is run ONCE on seed=42.
#  The selected configuration is used for all 10 seeds.
#
#  For each seed:
#    Step 1 — Split: 70% train / 30% test  (test held out entirely)
#    Step 2 — Scale: MinMaxScaler fitted on train only (no leakage)
#    Step 3 — Fit NGBoost on full training set with selected config
#             Early stopping uses a 10% internal val split
#    Step 4 — Evaluate on test set, collect all metrics
#
#  After all seeds → report mean ± std
# ============================================================

print("=" * 65)
print("  NGBoost — MULTI-SEED EVALUATION WITH 5-FOLD CV")
print("=" * 65)

# ── Step 0: Hyperparameter selection (once on seed=42) ───────
print("\n  Running hyperparameter selection on seed=42 split...")

set_seeds(42)
X_cv_raw, _, y_cv_raw, _ = train_test_split(X, y, test_size=0.30, random_state=42)
scaler_cv = MinMaxScaler()
X_cv_s    = scaler_cv.fit_transform(X_cv_raw)
y_cv      = (y_cv_raw.values if hasattr(y_cv_raw, "values")
             else np.array(y_cv_raw))

best_config_name, best_config, cv_summary = select_config_cv(X_cv_s, y_cv, seed=42)

print(f"\n  ✅ Configuration fixed for all seeds: {best_config_name}")
print(f"\n  CV Summary (supplementary material):")
print(cv_summary.to_string(index=False))

# ── Main seed loop ───────────────────────────────────────────
all_results     = []
all_predictions = []

for seed in SEEDS:
    print(f"\n{'='*65}")
    print(f"  SEED {seed}")
    print(f"{'='*65}")

    # ----------------------------------------------------------
    #  Step 1: Split
    #  70% train / 30% test. Test held out until Step 4.
    # ----------------------------------------------------------
    set_seeds(seed)
    X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
        X, y, test_size=0.30, random_state=seed
    )

    # ----------------------------------------------------------
    #  Step 2: Scale
    #  Scaler fitted on training data only — no leakage.
    # ----------------------------------------------------------
    scaler    = MinMaxScaler()
    X_train_s = scaler.fit_transform(X_train_raw)
    X_test_s  = scaler.transform(X_test_raw)

    y_train = (y_train_raw.values if hasattr(y_train_raw, "values")
               else np.array(y_train_raw))
    y_test  = (y_test_raw.values  if hasattr(y_test_raw,  "values")
               else np.array(y_test_raw))

    # ----------------------------------------------------------
    #  Step 3: Fit NGBoost on full training set
    #
    #  A 10% internal split drives early stopping only.
    #  This split is not used for any reported metric.
    #  Early stopping prevents the boosting from overfitting
    #  beyond the optimal number of trees.
    # ----------------------------------------------------------
    print(f"\n  Fitting NGBoost ({best_config_name}) on training set...")

    X_tr, X_es, y_tr, y_es = train_test_split(
        X_train_s, y_train, test_size=0.10, random_state=seed
    )

    set_seeds(seed)
    model = build_ngboost(best_config, seed=seed)

    model.fit(
        X_tr, y_tr,
        X_val=X_es,
        Y_val=y_es,
        early_stopping_rounds=50
    )

    n_trees_used = model.best_val_loss_itr if hasattr(model, "best_val_loss_itr") \
                   else best_config["n_estimators"]
    print(f"  Training complete. Estimators used: {n_trees_used}")

    # ----------------------------------------------------------
    #  Step 4: Test set evaluation
    #  First and only time the test set is used.
    #  NGBoost returns predictive distribution analytically.
    # ----------------------------------------------------------
    mu_test, sig_test = ngb_predict(model, X_test_s)

    r2      = r2_score(y_test, mu_test)
    mae     = mean_absolute_error(y_test, mu_test)
    mse     = mean_squared_error(y_test, mu_test)
    nll     = nll_gaussian(y_test, mu_test, sig_test)
    sharp   = float(np.mean(sig_test))
    ece     = compute_ece(y_test, mu_test, sig_test)
    cov, iw = compute_coverage_and_iw(y_test, mu_test, sig_test)
    miscal  = compute_miscal_area(y_test, mu_test, sig_test)

    all_results.append({
        "Seed":         seed,
        "Config":       best_config_name,
        "R²":           round(r2,    4),
        "MAE":          round(mae,   4),
        "MSE":          round(mse,   4),
        "NLL":          round(nll,   4),
        "Sharpness":    round(sharp, 4),
        "ECE":          round(ece,   4),
        "Coverage (%)": round(cov,   2),
        "IW":           round(iw,    4),
        "Miscal. Area": round(miscal,4),
    })

    all_predictions.append({
        "seed":     seed,
        "y_test":   y_test,
        "mu_test":  mu_test,
        "sig_test": sig_test,
    })

    print(f"\n  Test Results:")
    print(f"    R²={r2:.4f}  MAE={mae:.4f}  MSE={mse:.4f}")
    print(f"    NLL={nll:.4f}  Sharpness={sharp:.4f}  ECE={ece:.4f}")
    print(f"    Coverage={cov:.1f}%  IW={iw:.4f}  Miscal.={miscal:.4f}")


# ============================================================
#  SECTION 7 — AGGREGATE RESULTS
# ============================================================

METRIC_COLS = ["R²", "MAE", "MSE", "NLL", "Sharpness",
               "ECE", "Coverage (%)", "IW", "Miscal. Area"]

results_df = pd.DataFrame(all_results)
means      = results_df[METRIC_COLS].mean().round(4)
stds       = results_df[METRIC_COLS].std().round(4)

print(f"\n{'='*65}")
print("  AGGREGATED RESULTS ACROSS ALL SEEDS")
print(f"{'='*65}")

print("\n📋 Per-seed results:")
print(results_df.to_string(index=False))

print(f"\n📋 Configuration used across all {len(SEEDS)} seeds: {best_config_name}")

print("\n📄 Paper-ready summary — NGBoost  (Mean ± Std, n=10 seeds):")
paper_rows = {m: f"{means[m]:.3f} ± {stds[m]:.3f}" for m in METRIC_COLS}
paper_df   = pd.DataFrame.from_dict(paper_rows, orient="index",
                                    columns=["NGBoost"])
print(paper_df.to_string())

# Save for use in combined figures
ngb_all_predictions = all_predictions.copy()
ngb_results_df      = results_df.copy()
print("\n  ✅ Saved: ngb_all_predictions, ngb_results_df")


# ============================================================
#  SECTION 8 — VISUALISATIONS
#
#  Plot 1: Metric stability bars across seeds
#  Plot 2: Mean calibration curve with ±std band
#  Plot 3: Prediction intervals for representative seed
#  Plot 4: Feature importance
#          (NGBoost-specific — unique among your five models)
# ============================================================

plt.rcParams.update({
    "font.size":       8,
    "font.family":     "serif",
    "font.serif":      ["Times New Roman"],
    "axes.labelsize":  8,
    "axes.titlesize":  9,
    "legend.fontsize": 6,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "figure.dpi":      300,
    "savefig.dpi":     300,
    "pdf.fonttype":    42,
    "ps.fonttype":     42,
})


# ── Plot 1: Metric stability across seeds ───────────────────
fig, axes = plt.subplots(3, 3, figsize=(7.5, 6.5))
axes = axes.flatten()

for i, m in enumerate(METRIC_COLS):
    ax = axes[i]
    ax.bar(range(len(SEEDS)), results_df[m],
           color="teal", alpha=0.75, edgecolor="white")
    ax.axhline(means[m], color="crimson", linestyle="--", linewidth=1.5,
               label=f"Mean = {means[m]:.3f}")
    ax.fill_between([-0.5, len(SEEDS) - 0.5],
                    means[m] - stds[m], means[m] + stds[m],
                    alpha=0.15, color="crimson",
                    label=f"±1 SD = {stds[m]:.3f}")
    ax.set_title(m, fontweight="bold", fontsize=9, pad=5)
    ax.set_xlabel("Seed", fontsize=8)
    ax.set_ylabel("Value", fontsize=8)
    ax.set_xticks(range(len(SEEDS)))
    ax.set_xticklabels([str(s) for s in SEEDS], rotation=45, fontsize=7, ha="right")
    ax.grid(axis="y", alpha=0.25, linestyle="--", linewidth=0.5)
    if i == 0:
        ax.legend(loc="lower right", fontsize=6, frameon=True,
                  fancybox=False, edgecolor="black", handlelength=1.5)

plt.suptitle("NGBoost — Metric Stability Across 10 Random Seeds",
             fontsize=11, fontweight="bold", y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig("ngb_seed_stability.png", dpi=300, bbox_inches="tight")
plt.show()
print("\n✅ Saved: ngb_seed_stability.png")


# ── Plot 2: Mean calibration curve with ±std band ───────────

all_empirical = []

for pred in all_predictions:
    empirical_this_seed = []
    for c in NOMINAL_LEVELS:
        z   = scipy.stats.norm.ppf((1 + c) / 2)
        emp = np.mean(
            (pred["y_test"] >= pred["mu_test"] - z * pred["sig_test"]) &
            (pred["y_test"] <= pred["mu_test"] + z * pred["sig_test"])
        )
        empirical_this_seed.append(emp)
    all_empirical.append(empirical_this_seed)

all_empirical  = np.array(all_empirical)
mean_empirical = all_empirical.mean(axis=0)
std_empirical  = all_empirical.std(axis=0)
miscal_mean    = float(np.trapz(
    np.abs(mean_empirical - NOMINAL_LEVELS), NOMINAL_LEVELS
))

# Save for combined calibration plot
ngb_mean_empirical = mean_empirical.copy()
ngb_std_empirical  = std_empirical.copy()
print("  ✅ Saved: ngb_mean_empirical, ngb_std_empirical")

fig, ax = plt.subplots(figsize=(3.5, 3.5))
ax.fill_between(NOMINAL_LEVELS,
                mean_empirical - std_empirical,
                mean_empirical + std_empirical,
                alpha=0.20, color="teal",
                label=f"±1 SD across {len(SEEDS)} seeds")
ax.plot(NOMINAL_LEVELS, mean_empirical, "o-", color="teal",
        linewidth=1.5, markersize=3,
        label=f"NGBoost (mean, n={len(SEEDS)} seeds)")
ax.plot([0, 1], [0, 1], "k--", linewidth=1.0, label="Ideal calibration")
ax.set_xlabel("Nominal confidence level", fontsize=9)
ax.set_ylabel("Empirical coverage", fontsize=9)
ax.set_title("Calibration Curve — NGBoost", fontsize=10, fontweight="bold")
ax.legend(fontsize=7, loc="lower right")
ax.text(0.05, 0.88,
        f"Miscalibration area = {miscal_mean:.4f}",
        transform=ax.transAxes, fontsize=8,
        bbox=dict(boxstyle="round,pad=0.3",
                  facecolor="lightyellow", alpha=0.8))
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("ngb_calibration.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Saved: ngb_calibration.png  "
      f"(mean miscalibration area = {miscal_mean:.4f})")


# ── Plot 3: Prediction intervals — representative seed ───────

mean_nll  = results_df["NLL"].mean()
rep_idx   = int(np.argmin(np.abs(results_df["NLL"].values - mean_nll)))
rep_seed  = int(results_df.iloc[rep_idx]["Seed"])
rep_nll   = results_df.iloc[rep_idx]["NLL"]
rep_cov   = results_df.iloc[rep_idx]["Coverage (%)"]

rep       = all_predictions[rep_idx]
y_rep     = rep["y_test"]
mu_rep    = rep["mu_test"]
sig_rep   = rep["sig_test"]
sort_idx  = np.argsort(y_rep)

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.fill_between(range(len(y_rep)),
                mu_rep[sort_idx] - 1.96 * sig_rep[sort_idx],
                mu_rep[sort_idx] + 1.96 * sig_rep[sort_idx],
                alpha=0.25, color="teal",
                label="95% Prediction Interval")
ax.plot(range(len(y_rep)), mu_rep[sort_idx],
        "o", markersize=3, color="teal", alpha=0.7,
        label="Predicted mean")
ax.plot(range(len(y_rep)), y_rep[sort_idx],
        "r-", linewidth=1.0, label="Actual values")
ax.set_xlabel("Test samples (sorted by actual value)", fontsize=9)
ax.set_ylabel("Log rupture time", fontsize=9)
ax.set_title(
    f"NGBoost — Predictions with 95% Intervals\n"
    f"Representative seed {rep_seed}  "
    f"(NLL = {rep_nll:.3f} ≈ mean NLL = {mean_nll:.3f},  "
    f"Coverage = {rep_cov:.1f}%)",
    fontsize=9, fontweight="bold"
)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("ngb_predictions.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"✅ Saved: ngb_predictions.png  (representative seed = {rep_seed})")


# ── Plot 4: Feature importance ────────────────────────────────
#
#  This plot is unique to NGBoost among your five models.
#  NGBoost's base learners are decision trees, so feature
#  importance is defined as the total reduction in the scoring
#  rule (NLL) attributable to each feature across all trees
#  and both distribution parameters (mean and variance).
#
#  Two importance scores are available:
#    loc_importances   → features important for predicting μ (mean)
#    scale_importances → features important for predicting σ (variance)
#
#  Comparing these two tells you whether the features that
#  drive point predictions are the same ones that drive
#  uncertainty — a physically meaningful finding.
#
#  The model from the representative seed is used for this plot,
#  consistent with Plot 3. Feature names come from the original
#  DataFrame X.
#
#  Re-fit the representative seed model (cleared from memory above)

print(f"\n  Re-fitting representative seed {rep_seed} for feature importance...")

set_seeds(rep_seed)
X_rep_raw, X_test_rep, y_rep_raw, _ = train_test_split(
    X, y, test_size=0.30, random_state=rep_seed
)
scaler_rep = MinMaxScaler()
X_rep_s    = scaler_rep.fit_transform(X_rep_raw)
y_rep_arr  = (y_rep_raw.values if hasattr(y_rep_raw, "values")
              else np.array(y_rep_raw))

X_rep_tr, X_rep_es, y_rep_tr, y_rep_es = train_test_split(
    X_rep_s, y_rep_arr, test_size=0.10, random_state=rep_seed
)

rep_model = build_ngboost(best_config, seed=rep_seed)
rep_model.fit(
    X_rep_tr, y_rep_tr,
    X_val=X_rep_es,
    Y_val=y_rep_es,
    early_stopping_rounds=50
)

feature_names = list(X.columns)

# loc_importances: importance for predicting the mean
# scale_importances: importance for predicting the std / variance
loc_imp   = rep_model.feature_importances_[0]   # mean component
scale_imp = rep_model.feature_importances_[1]   # variance component

# Normalise to sum to 1 for comparability
loc_imp   = loc_imp   / loc_imp.sum()
scale_imp = scale_imp / scale_imp.sum()

# Sort by loc importance for consistent ordering
sort_feat = np.argsort(loc_imp)[::-1]
top_n     = min(15, len(feature_names))   # show top 15 features

fig, axes = plt.subplots(1, 2, figsize=(7.5, 5))

# Mean importance
axes[0].barh(
    range(top_n),
    loc_imp[sort_feat[:top_n]][::-1],
    color="teal", alpha=0.75, edgecolor="white"
)
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(
    [feature_names[i] for i in sort_feat[:top_n]][::-1],
    fontsize=7
)
axes[0].set_xlabel("Normalised importance", fontsize=8)
axes[0].set_title("(a) Mean prediction\n(loc component)",
                  fontsize=9, fontweight="bold")
axes[0].grid(axis="x", alpha=0.25, linestyle="--")

# Variance importance
sort_scale = np.argsort(scale_imp)[::-1]
axes[1].barh(
    range(top_n),
    scale_imp[sort_scale[:top_n]][::-1],
    color="coral", alpha=0.75, edgecolor="white"
)
axes[1].set_yticks(range(top_n))
axes[1].set_yticklabels(
    [feature_names[i] for i in sort_scale[:top_n]][::-1],
    fontsize=7
)
axes[1].set_xlabel("Normalised importance", fontsize=8)
axes[1].set_title("(b) Uncertainty prediction\n(scale component)",
                  fontsize=9, fontweight="bold")
axes[1].grid(axis="x", alpha=0.25, linestyle="--")

plt.suptitle(
    f"NGBoost — Feature Importance\n"
    f"(Representative seed {rep_seed}; "
    f"top {top_n} of {len(feature_names)} features)",
    fontsize=9, fontweight="bold"
)
plt.tight_layout()
plt.savefig("ngb_feature_importance.png", dpi=300, bbox_inches="tight")
plt.show()
print("✅ Saved: ngb_feature_importance.png")

# Print top 5 features for each component
print("\n📋 Top 5 features for mean prediction (loc):")
for rank, i in enumerate(sort_feat[:5], 1):
    print(f"  {rank}. {feature_names[i]:<35}  {loc_imp[i]:.4f}")

print("\n📋 Top 5 features for uncertainty prediction (scale):")
for rank, i in enumerate(sort_scale[:5], 1):
    print(f"  {rank}. {feature_names[i]:<35}  {scale_imp[i]:.4f}")

# Overlap analysis: are the same features driving mean and uncertainty?
top5_loc   = set(sort_feat[:5])
top5_scale = set(sort_scale[:5])
overlap    = top5_loc & top5_scale
print(f"\n  Features in top 5 for BOTH mean and uncertainty: "
      f"{len(overlap)}/5")
if overlap:
    print(f"  Overlapping features: "
          f"{[feature_names[i] for i in overlap]}")
else:
    print("  No overlap — different features drive mean vs uncertainty.")

# Save calibration data for combined plot
ngb_mean_empirical = mean_empirical.copy()
ngb_std_empirical  = std_empirical.copy()

# Save predictions and results for combined uncertainty plot
ngb_all_predictions = all_predictions.copy()
ngb_results_df      = results_df.copy()

# Save to disk
import joblib
joblib.dump(ngb_mean_empirical, 'ngb_mean_empirical.pkl')
joblib.dump(ngb_std_empirical, 'ngb_std_empirical.pkl')
joblib.dump(ngb_all_predictions, 'ngb_all_predictions.pkl')
joblib.dump(ngb_results_df, 'ngb_results_df.pkl')
print("✅ NGBoost data saved to disk")

print(f"\n{'='*65}")
print("  NGBoost — COMPLETE")
print(f"{'='*65}")